# IFRS S1/S2 Report Engine — v2 (rebuilt)

Rebuilt per the signed-off audit (`IFRS_pipeline_audit_report.md`). Core inversion vs v1:
**facts are bound before writing; coverage is verified after writing.**

**Pipeline:** payload profile & arithmetic audit → requirement applicability (predicates + LLM for
conditional clauses) → evidence binding (lexical proposer + LLM verifier, cached) → typed section IR
with a **fact table** (every number carries path, unit, period) → writer emits numbers only as
`{{FACT:key}}` tokens → deterministic rendering & gates (token substitution, literal numbers, period
consistency, style mechanics, no-copy firewall, cleanliness) → LLM output-coverage verifier
(per-requirement, quote-checked) + qualitative-claims check + style judge → targeted span-level
repair (requirement patches, sentence fixes) → approval constitution (fixed, waiver-logged) →
assembly → final gates on the shipped bytes → independent scorecard.

**Data constraint honoured:** requirements that are applicable but unsupported by the payload live in
audit-only registers and can never fail approval or lower a score. "Missing" strictly means:
applicable + evidence exists + the report failed to use it.

**Run configuration (env vars):**
- `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_GPT52_DEPLOYMENT_URL`, `AZURE_OPENAI_FAST_DEPLOYMENT_URL` — same as v1.
- `IFRS_LLM_MODE` = `azure` (default) | `mock` (offline smoke test of the whole pipeline).
- `IFRS_BASE_DIR` — folder containing `payload_BANK01_v2.json` and the five `*_requirements.json` files
  (defaults to the notebook's directory). Optional `style_system/` folder enables the ablation arms.
- `IFRS_STYLE_ARM` = `none` (default, per sign-off) | `raw` | `refined`; `IFRS_RUN_STYLE_ABLATION=1` runs all three.
- `IFRS_MAX_REPAIR_LOOPS` (default 3).

Outputs land in `run_outputs/00_…09_` stage folders; every stage checkpoints to disk; sections that
cannot be approved escalate to `human_review_*.md` with the exact unresolved requirement IDs.


## 1 · Configuration & IO

In [ ]:
# ============================================================
# CELL 1 — CONFIGURATION, PATHS, IO
# One definition per function. No cell in this notebook redefines
# a function declared in an earlier cell.
# ============================================================
import os, re, json, math, hashlib, time, unicodedata
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from collections import Counter, defaultdict
from decimal import Decimal, InvalidOperation

# ---- run modes -------------------------------------------------------------
# LLM_MODE:
#   "azure" — real Azure OpenAI calls (production)
#   "mock"  — deterministic stand-ins for every LLM stage; lets the whole
#             pipeline run offline so the deterministic machinery is testable.
LLM_MODE = os.getenv("IFRS_LLM_MODE", "azure").strip().lower()

# STYLE_ARM: "none" (built-in contract only; per sign-off this is the default),
#            "raw" (inject extracted style verbatim), "refined" (refined style).
STYLE_ARM = os.getenv("IFRS_STYLE_ARM", "none").strip().lower()

MAX_REPAIR_LOOPS = int(os.getenv("IFRS_MAX_REPAIR_LOOPS", "3"))



CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    NOTEBOOK_DIR = CURRENT_DIR
elif (CURRENT_DIR / "notebooks").exists():
    NOTEBOOK_DIR = (CURRENT_DIR / "notebooks").resolve()
else:
    NOTEBOOK_DIR = CURRENT_DIR

BASE_DIR = NOTEBOOK_DIR / "gen_data"



# ---- paths -----------------------------------------------------------------
PAYLOAD_PATH = Path(os.getenv("IFRS_PAYLOAD_PATH", str(BASE_DIR / "payloads_risk" / "payload_BANK01_v2.json")))
REQUIREMENTS_DIR = Path(os.getenv("IFRS_REQUIREMENTS_DIR", str(BASE_DIR / "IFRS" / "ifrs_requirements_kb_outputs_final" / "section_by_section_requirements" / "json")))
STYLE_SYSTEM_DIR = Path(os.getenv("IFRS_STYLE_SYSTEM_DIR", str(BASE_DIR / "style" / "style_system")))
OUTPUT_DIR = Path(os.getenv("IFRS_OUTPUT_DIR", str(BASE_DIR / "run_outputs")))

STAGE_DIRS = {
    "audit":        OUTPUT_DIR / "00_payload_audit",
    "applicability":OUTPUT_DIR / "01_applicability",
    "bindings":     OUTPUT_DIR / "02_evidence_bindings",
    "ir":           OUTPUT_DIR / "03_section_ir",
    "drafts":       OUTPUT_DIR / "04_drafts",
    "verification": OUTPUT_DIR / "05_verification",
    "repairs":      OUTPUT_DIR / "06_repairs",
    "approved":     OUTPUT_DIR / "07_approved_sections",
    "assembly":     OUTPUT_DIR / "08_assembled_report",
    "scores":       OUTPUT_DIR / "09_scores",
    "logs":         OUTPUT_DIR / "logs",
    "cache":        OUTPUT_DIR / "cache",
}
for d in STAGE_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

SECTIONS = ["General Requirements", "Governance", "Strategy", "Risk Management", "Metrics and Targets"]
SECTION_SLUGS = {
    "General Requirements": "general_requirements",
    "Governance": "governance",
    "Strategy": "strategy",
    "Risk Management": "risk_management",
    "Metrics and Targets": "metrics_and_targets",
}
SECTION_NUMBERS = {s: i + 1 for i, s in enumerate(SECTIONS)}

# Deterministic routing: which payload roots each section may draw evidence from.
SECTION_PAYLOAD_ROOTS = {
    "General Requirements": ["general_requirements_context", "metadata", "bank", "financial_summary",
                             "ghg_methodology", "scope12_consolidation"],
    "Governance": ["governance", "board_minutes", "reporting_kpis", "bank"],
    "Strategy": ["climate_scenarios", "climate_risk_register", "climate_opportunities", "value_chain_map",
                 "transition_plan", "climate_financial_effects", "resilience_assessment", "bank",
                 "financial_summary", "reporting_kpis"],
    "Risk Management": ["climate_risk_register", "physical_risk_exposures", "climate_scenarios",
                        "value_chain_map", "governance", "reporting_kpis"],
    "Metrics and Targets": ["scope1", "scope2", "scope3_travel", "scope3_categories", "financed_emissions",
                            "financed_emissions_equity", "financed_emissions_sovereign", "targets",
                            "carbon_credits", "internal_carbon_price", "ghg_methodology",
                            "scope12_consolidation", "reporting_kpis", "metadata"],
}

# evidence_tags → payload roots (deterministic, dataset-agnostic in spirit:
# the tags come from the requirements KB; only the right side names this payload's roots).
TAG_TO_ROOTS = {
    "governance_body": ["governance", "board_minutes"],
    "management_role": ["governance", "board_minutes"],
    "remuneration": ["governance"],
    "risk_process": ["climate_risk_register", "physical_risk_exposures", "governance"],
    "scenario_analysis": ["climate_scenarios", "resilience_assessment"],
    "business_model_value_chain": ["value_chain_map", "bank", "climate_opportunities"],
    "strategy_decision_making": ["transition_plan", "climate_opportunities", "climate_financial_effects",
                                 "internal_carbon_price"],
    "financial_effects": ["climate_financial_effects", "financial_summary", "reporting_kpis"],
    "materiality": ["climate_risk_register", "value_chain_map", "general_requirements_context"],
    "connected_information": ["general_requirements_context", "financial_summary", "climate_financial_effects"],
    "metrics": ["reporting_kpis", "scope1", "scope2", "scope3_categories", "financed_emissions",
                "internal_carbon_price"],
    "targets": ["targets", "transition_plan", "reporting_kpis"],
    "ghg_emissions": ["scope1", "scope2", "scope3_travel", "scope3_categories", "ghg_methodology",
                      "scope12_consolidation", "reporting_kpis"],
    "scope_1": ["scope1", "ghg_methodology", "scope12_consolidation"],
    "scope_2": ["scope2", "ghg_methodology", "scope12_consolidation"],
    "scope_3": ["scope3_categories", "scope3_travel", "financed_emissions", "ghg_methodology"],
    "financed_emissions": ["financed_emissions", "financed_emissions_equity", "financed_emissions_sovereign"],
    "carbon_credits": ["carbon_credits"],
    "commercial_banking": ["financed_emissions", "financial_summary", "reporting_kpis"],
    "asset_management": ["financed_emissions_equity", "financed_emissions_sovereign"],
    "insurance": [],           # entity is a bank; insurance-tagged clauses resolved by applicability
    "source_guidance": [],     # guidance references, not disclosure evidence
}

# ---- small IO helpers ------------------------------------------------------
def read_json(path: Path, default: Any = None) -> Any:
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return default

def write_json(obj: Any, path: Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def write_text(text: str, path: Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(text, encoding="utf-8")

def read_text(path: Path, default: str = "") -> str:
    try:
        return Path(path).read_text(encoding="utf-8")
    except Exception:
        return default

def sha12(obj: Any) -> str:
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode("utf-8")).hexdigest()[:12]

def flatten_json(obj: Any, prefix: str = "") -> Dict[str, Any]:
    """Flatten nested payload into {dotted.path[index].leaf: scalar}."""
    out: Dict[str, Any] = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f"{prefix}.{k}" if prefix else str(k)
            out.update(flatten_json(v, p))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            out.update(flatten_json(v, f"{prefix}[{i}]"))
    else:
        out[prefix] = obj
    return out

def root_of_path(path: str) -> str:
    return re.split(r"[.\[]", str(path), 1)[0]

def get_by_path(obj: Any, path: str) -> Any:
    """Resolve 'a.b[2].c' against nested data; None if unresolvable."""
    try:
        cur = obj
        for part in re.findall(r"[^.\[\]]+|\[\d+\]", str(path)):
            if part.startswith("["):
                cur = cur[int(part[1:-1])]
            else:
                cur = cur[part]
        return cur
    except Exception:
        return None

RUN_STAMP = time.strftime("%Y%m%d_%H%M%S")
print(f"Config ready | LLM_MODE={LLM_MODE} | STYLE_ARM={STYLE_ARM} | outputs -> {OUTPUT_DIR}")


## 2 · LLM client (Azure / mock) — every call logged, no silent failures

In [ ]:
# ============================================================
# CELL 2 — ROBUST AZURE OPENAI CLIENT
# Replaces the engine notebook's current Cell 2
# ============================================================

import json
import os
import random
import re
import time
import urllib.error
import urllib.request

from typing import Any, Dict, Optional


# ============================================================
# Azure configuration
# ============================================================

AZURE_OPENAI_API_KEY = (
    os.getenv("AZURE_OPENAI_API_KEY")
    or os.getenv("OPENAI_API_KEY")
)


def _clean_url(value: Optional[str]) -> Optional[str]:
    if not value:
        return None

    value = str(value).strip().strip('"').strip("'")

    markdown_match = re.search(
        r"\]\((https://[^)\s]+)\)",
        value,
    )

    if markdown_match:
        value = markdown_match.group(1).strip()

    https_positions = [
        match.start()
        for match in re.finditer(r"https://", value)
    ]

    if https_positions:
        value = value[https_positions[-1]:]

    return value.strip().strip("[]").rstrip(").,;")


AZURE_STRONG_URL = _clean_url(
    os.getenv("AZURE_OPENAI_GPT52_DEPLOYMENT_URL")
)

AZURE_FAST_URL = (
    _clean_url(
        os.getenv("AZURE_OPENAI_FAST_DEPLOYMENT_URL")
    )
    or AZURE_STRONG_URL
)


# ============================================================
# Model routing
# ============================================================

MODEL_TIERS = {
    "writer": "strong",
    "patch_writer": "strong",
    "binding_verifier": "fast",
    "applicability_assessor": "fast",
    "coverage_verifier": "strong",
    "claims_checker": "strong",
    "style_judge": "fast",
    "editorial": "strong",
}


# ============================================================
# Validate configuration
# ============================================================

def validate_llm_config() -> None:
    if LLM_MODE == "mock":
        print("LLM_MODE=mock: skipping Azure configuration validation.")
        return
    missing = []

    if not AZURE_OPENAI_API_KEY:
        missing.append("AZURE_OPENAI_API_KEY")

    if not AZURE_STRONG_URL:
        missing.append(
            "AZURE_OPENAI_GPT52_DEPLOYMENT_URL"
        )

    if not AZURE_FAST_URL:
        missing.append(
            "AZURE_OPENAI_FAST_DEPLOYMENT_URL"
        )

    if missing:
        raise ValueError(
            "Missing Azure configuration:\n"
            + "\n".join(f"- {name}" for name in missing)
        )

    for name, url in {
        "strong": AZURE_STRONG_URL,
        "fast": AZURE_FAST_URL,
    }.items():
        if not url.startswith("https://"):
            raise ValueError(
                f"The {name} Azure URL must start with https://"
            )

        if "/chat/completions" not in url:
            raise ValueError(
                f"The {name} Azure URL must contain "
                f"'/chat/completions'."
            )


validate_llm_config()


# ============================================================
# LLM logging
# ============================================================

_LLM_CALL_LOG = STAGE_DIRS["logs"] / "llm_calls.jsonl"
_LLM_CALL_LOG.parent.mkdir(parents=True, exist_ok=True)


def _log_llm(record: Dict[str, Any]) -> None:
    with open(
        _LLM_CALL_LOG,
        "a",
        encoding="utf-8",
    ) as file:
        file.write(
            json.dumps(
                record,
                ensure_ascii=False,
                default=str,
            )
            + "\n"
        )


# ============================================================
# Azure HTTP errors
# ============================================================

class AzureHTTPError(RuntimeError):
    def __init__(
        self,
        status_code: int,
        response_body: str,
        headers: Optional[Dict[str, str]] = None,
    ):
        self.status_code = status_code
        self.response_body = response_body
        self.headers = headers or {}

        super().__init__(
            f"Azure HTTP {status_code}:\n"
            f"{response_body[:3000]}"
        )


def _azure_request(
    url: str,
    body: Dict[str, Any],
    timeout: int = 240,
) -> Dict[str, Any]:

    request = urllib.request.Request(
        url,
        data=json.dumps(body).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "api-key": AZURE_OPENAI_API_KEY or "",
        },
        method="POST",
    )

    try:
        with urllib.request.urlopen(
            request,
            timeout=timeout,
        ) as response:
            return json.loads(
                response.read().decode("utf-8")
            )

    except urllib.error.HTTPError as exc:
        response_body = exc.read().decode(
            "utf-8",
            errors="replace",
        )

        headers = (
            dict(exc.headers.items())
            if exc.headers
            else {}
        )

        raise AzureHTTPError(
            status_code=exc.code,
            response_body=response_body,
            headers=headers,
        ) from exc


# ============================================================
# Response helpers
# ============================================================

def _strip_json_fences(text: str) -> str:
    text = str(text or "").strip()

    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\s*```$",
        "",
        text,
    )

    return text.strip()


def _parse_json_content(
    content: str,
    role_label: str,
) -> Dict[str, Any]:

    candidate = _strip_json_fences(content)

    try:
        return json.loads(candidate)

    except json.JSONDecodeError:
        # Handle short commentary before or after the JSON.
        first_brace = candidate.find("{")
        last_brace = candidate.rfind("}")

        if first_brace >= 0 and last_brace > first_brace:
            extracted = candidate[
                first_brace:last_brace + 1
            ]

            try:
                return json.loads(extracted)
            except json.JSONDecodeError:
                pass

        raise ValueError(
            f"LLM call '{role_label}' returned invalid JSON.\n"
            f"Response preview:\n{candidate[:2000]}"
        )


def _retry_after_seconds(
    headers: Dict[str, str],
) -> Optional[float]:

    value = (
        headers.get("Retry-After")
        or headers.get("retry-after")
    )

    if value is None:
        return None

    try:
        return float(value)
    except (TypeError, ValueError):
        return None


# ============================================================
# Main engine LLM function
# ============================================================

def llm_json(
    role_label: str,
    system: str,
    user: str,
    max_tokens: int = 4000,
    temperature: Optional[float] = None,
) -> Dict[str, Any]:
    """
    JSON-returning Azure call used by the engine.

    The temperature argument remains in the signature so existing
    engine calls do not need to change. It is deliberately not sent,
    because some GPT-5/reasoning deployments reject temperature.
    """

    started_at = time.time()

    # --------------------------------------------------------
    # Mock mode
    # --------------------------------------------------------

    if LLM_MODE == "mock":
        handler_name = role_label.split(":")[0]
        handler = MOCK_HANDLERS.get(handler_name)

        if handler is None:
            raise RuntimeError(
                f"No mock handler for LLM role '{role_label}'"
            )

        output = handler(system, user)

        _log_llm({
            "role": role_label,
            "mode": "mock",
            "latency_s": round(
                time.time() - started_at,
                3,
            ),
        })

        return output

    # --------------------------------------------------------
    # Azure routing
    # --------------------------------------------------------

    role_name = role_label.split(":")[0]

    tier = MODEL_TIERS.get(
        role_name,
        "strong",
    )

    url = (
        AZURE_STRONG_URL
        if tier == "strong"
        else AZURE_FAST_URL
    )

    if not url or not AZURE_OPENAI_API_KEY:
        raise RuntimeError(
            "Azure URL or API key is not configured."
        )

    base_body = {
        "messages": [
            {
                "role": "system",
                "content": system,
            },
            {
                "role": "user",
                "content": user,
            },
        ],
        "response_format": {
            "type": "json_object",
        },
    }

    # Preferred field first; legacy fallback second.
    token_fields = [
        "max_completion_tokens",
        "max_tokens",
    ]

    max_attempts = 6
    last_error: Optional[Exception] = None

    for token_field in token_fields:
        body = dict(base_body)
        body[token_field] = max_tokens

        for attempt in range(1, max_attempts + 1):
            try:
                data = _azure_request(
                    url=url,
                    body=body,
                )

                try:
                    content = data[
                        "choices"
                    ][0][
                        "message"
                    ][
                        "content"
                    ]
                except (KeyError, IndexError, TypeError) as exc:
                    raise ValueError(
                        "Unexpected Azure response:\n"
                        + json.dumps(
                            data,
                            ensure_ascii=False,
                            indent=2,
                        )[:3000]
                    ) from exc

                result = _parse_json_content(
                    content=content,
                    role_label=role_label,
                )

                _log_llm({
                    "role": role_label,
                    "mode": "azure",
                    "tier": tier,
                    "token_field": token_field,
                    "attempt": attempt,
                    "latency_s": round(
                        time.time() - started_at,
                        3,
                    ),
                    "prompt_chars": len(system) + len(user),
                    "completion_chars": len(content),
                    "usage": data.get("usage", {}),
                })

                return result

            except AzureHTTPError as exc:
                last_error = exc

                _log_llm({
                    "role": role_label,
                    "mode": "azure",
                    "tier": tier,
                    "token_field": token_field,
                    "attempt": attempt,
                    "status_code": exc.status_code,
                    "response": exc.response_body[:1500],
                })

                # Request compatibility error.
                if exc.status_code in {400, 422}:
                    if token_field == "max_completion_tokens":
                        print(
                            f"{role_label}: "
                            "`max_completion_tokens` was rejected; "
                            "trying `max_tokens`."
                        )
                        break

                    raise RuntimeError(
                        f"LLM call '{role_label}' was rejected.\n"
                        f"{exc}"
                    ) from exc

                # Rate limit.
                if exc.status_code == 429:
                    if attempt < max_attempts:
                        retry_after = _retry_after_seconds(
                            exc.headers
                        )

                        wait = (
                            retry_after
                            if retry_after is not None
                            else min(
                                (2 ** attempt) + random.random(),
                                60,
                            )
                        )

                        print(
                            f"{role_label}: rate limited; "
                            f"retrying in {wait:.1f}s."
                        )

                        time.sleep(wait)
                        continue

                    raise

                # Temporary Azure/server errors.
                if exc.status_code in {
                    500,
                    502,
                    503,
                    504,
                }:
                    if attempt < max_attempts:
                        wait = min(
                            2 ** (attempt - 1)
                            + random.random(),
                            12,
                        )

                        print(
                            f"{role_label}: Azure error "
                            f"{exc.status_code}; retrying "
                            f"in {wait:.1f}s."
                        )

                        time.sleep(wait)
                        continue

                    raise

                raise

            except (
                urllib.error.URLError,
                TimeoutError,
                ConnectionError,
                OSError,
            ) as exc:
                last_error = exc

                _log_llm({
                    "role": role_label,
                    "mode": "azure",
                    "tier": tier,
                    "token_field": token_field,
                    "attempt": attempt,
                    "error": repr(exc)[:1500],
                })

                if attempt < max_attempts:
                    wait = min(
                        2 ** (attempt - 1)
                        + random.random(),
                        12,
                    )

                    print(
                        f"{role_label}: connection problem; "
                        f"retrying in {wait:.1f}s."
                    )

                    time.sleep(wait)
                    continue

                raise RuntimeError(
                    f"LLM call '{role_label}' failed because "
                    f"of a connection problem: {exc}"
                ) from exc

    raise RuntimeError(
        f"LLM call '{role_label}' failed.\n"
        f"Last Azure error:\n{last_error}"
    )


# ============================================================
# Prompt truncation — required by later engine cells
# ============================================================

def truncate_for_prompt(
    obj: Any,
    max_chars: int = 60000,
) -> str:

    text = json.dumps(
        obj,
        ensure_ascii=False,
        indent=1,
        default=str,
    )

    if len(text) <= max_chars:
        return text

    return (
        text[:max_chars - 60]
        + "\n...[truncated for prompt length]..."
    )


# Later cells register the mock handlers.
MOCK_HANDLERS: Dict[str, Any] = {}


# ============================================================
# Connection test
# ============================================================

def test_engine_llm(
    tier: str = "strong",
) -> Dict[str, Any]:

    role_label = (
        "writer:connection_test"
        if tier == "strong"
        else "style_judge:connection_test"
    )

    result = llm_json(
        role_label=role_label,
        system=(
            "Return one valid JSON object only. "
            "Do not use markdown."
        ),
        user='Return {"status": "ok"}.',
        max_tokens=500,
    )

    print(f"{tier} endpoint response:", result)
    return result


print(
    "LLM client ready | "
    f"mode={LLM_MODE} | "
    f"strong_url={'set' if AZURE_STRONG_URL else 'MISSING'} | "
    f"fast_url={'set' if AZURE_FAST_URL else 'MISSING'}"
)

## 3 · Load payload & per-section requirements (prohibitions split out)

In [ ]:
# ============================================================
# CELL 3 — LOAD PAYLOAD AND REQUIREMENTS
# Requirements come from the five per-section *_requirements.json files
# (no separate KB files). Prohibition-type clauses ("shall not") are split
# out: they become writer/verifier CONSTRAINTS, not coverage rows.
# ============================================================
PAYLOAD: Dict[str, Any] = read_json(PAYLOAD_PATH)
if not PAYLOAD:
    raise FileNotFoundError(f"Payload not found or empty: {PAYLOAD_PATH}")

FLAT_PAYLOAD: Dict[str, Any] = flatten_json(PAYLOAD)
REPORTING_YEAR = int(get_by_path(PAYLOAD, "metadata.reporting_year") or
                     get_by_path(PAYLOAD, "general_requirements_context.reporting_year") or 0)
COMPARATIVE_YEARS = [int(y) for y in (get_by_path(PAYLOAD, "metadata.comparative_years") or [])]
ALL_YEARS = sorted(set([REPORTING_YEAR] + COMPARATIVE_YEARS))
ENTITY_NAME = str(get_by_path(PAYLOAD, "general_requirements_context.reporting_entity")
                  or get_by_path(PAYLOAD, "bank.bank_name") or "the reporting entity")

_REQ_FILES = {s: REQUIREMENTS_DIR / f"{SECTION_SLUGS[s]}_requirements.json" for s in SECTIONS}

def _load_section_requirements(section: str) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    doc = read_json(_REQ_FILES[section])
    if not doc:
        raise FileNotFoundError(f"Requirements file missing for {section}: {_REQ_FILES[section]}")
    reqs, prohibitions = [], []
    for std_name, block in (doc.get("standards") or {}).items():
        for r in block.get("requirements", []):
            r = dict(r)
            r["section_name"] = section
            (prohibitions if r.get("obligation_type") == "prohibition" else reqs).append(r)
    return reqs, prohibitions

requirements_by_section: Dict[str, List[Dict[str, Any]]] = {}
prohibitions_by_section: Dict[str, List[Dict[str, Any]]] = {}
for _s in SECTIONS:
    requirements_by_section[_s], prohibitions_by_section[_s] = _load_section_requirements(_s)

REQ_INDEX: Dict[str, Dict[str, Any]] = {
    r["requirement_id"]: r for s in SECTIONS for r in requirements_by_section[s]
}

print(f"Payload: {len(FLAT_PAYLOAD)} leaves | entity: {ENTITY_NAME} | reporting year {REPORTING_YEAR} "
      f"| comparatives {COMPARATIVE_YEARS}")
for _s in SECTIONS:
    print(f"  {_s}: {len(requirements_by_section[_s])} disclosure requirements, "
          f"{len(prohibitions_by_section[_s])} prohibitions")


## 4 · Payload profile + deterministic arithmetic audit (pre-flight)

In [ ]:
# ============================================================
# CELL 4 — PAYLOAD PROFILE + DETERMINISTIC ARITHMETIC AUDIT (pre-flight)
# Rebuilt generic version of the previous notebook's arithmetic auditor:
# schema-discovering, per-year, tolerance-aware. Findings are recorded and
# hard-fail only on exact internal contradictions; near-misses are warnings.
# ============================================================
_ARITH_REL_TOL = float(os.getenv("IFRS_ARITH_REL_TOL", "0.005"))   # 0.5%

def _to_dec(v: Any) -> Optional[Decimal]:
    try:
        if isinstance(v, bool) or v is None:
            return None
        d = Decimal(str(v))
        return d if d.is_finite() else None
    except (InvalidOperation, ValueError):
        return None

def _rel_diff(a: Decimal, b: Decimal) -> float:
    denom = max(abs(a), abs(b), Decimal("1e-9"))
    return float(abs(a - b) / denom)

def build_payload_profile() -> Dict[str, Any]:
    """What exists in this payload — drives applicability predicates."""
    def rows(root):  # list-of-dict roots
        v = PAYLOAD.get(root)
        return v if isinstance(v, list) else ([] if v is None else [v])
    scope3_rows = rows("scope3_categories")
    profile = {
        "entity_name": ENTITY_NAME,
        "reporting_year": REPORTING_YEAR,
        "comparative_years": COMPARATIVE_YEARS,
        "has_comparatives": bool(COMPARATIVE_YEARS),
        "currency": get_by_path(PAYLOAD, "general_requirements_context.reporting_currency"),
        "boundary_type": get_by_path(PAYLOAD, "general_requirements_context.boundary_type"),
        "has_data_gaps": bool(get_by_path(PAYLOAD, "metadata.data_gaps")),
        "data_gap_count": len(get_by_path(PAYLOAD, "metadata.data_gaps") or []),
        "external_assurance": get_by_path(PAYLOAD, "general_requirements_context.external_assurance"),
        "has_scope3_categories": any(r.get("included_flag") for r in scope3_rows),
        "scope3_included_categories": sorted({int(r["category_number"]) for r in scope3_rows
                                              if r.get("included_flag") and r.get("category_number") is not None}),
        "scope3_excluded_with_reason": sum(1 for r in scope3_rows
                                           if not r.get("included_flag") and r.get("exclusion_reason")),
        "has_financed_emissions": bool(rows("financed_emissions")),
        "has_carbon_credits": bool(rows("carbon_credits")),
        "has_internal_carbon_price": bool(rows("internal_carbon_price")),
        "has_targets": bool(rows("targets")),
        "target_frameworks": sorted({str(t.get("target_framework")) for t in rows("targets") if t.get("target_framework")}),
        "has_scenario_analysis": bool(rows("climate_scenarios")),
        "scenario_frameworks": sorted({str(s.get("framework")) for s in rows("climate_scenarios") if s.get("framework")}),
        "has_transition_plan": any(t.get("has_transition_plan") for t in rows("transition_plan")),
        "has_remuneration_link": any(g.get("ceo_compensation_esg_linked") for g in rows("governance")),
        "has_physical_risk_data": bool(rows("physical_risk_exposures")),
        "has_value_chain_map": bool(rows("value_chain_map")),
        "has_climate_financial_effects": bool(rows("climate_financial_effects")),
        "has_resilience_assessment": bool(rows("resilience_assessment")),
        "business_lines": {
            "commercial_banking": bool(rows("financed_emissions")),
            "asset_management": bool(rows("financed_emissions_equity")) or bool(rows("financed_emissions_sovereign")),
            "insurance": False,  # no insurance data roots exist in this payload
        },
        "payload_roots_present": sorted([k for k in PAYLOAD.keys()]),
    }
    return profile

def run_payload_arithmetic_audit() -> Dict[str, Any]:
    issues: List[Dict[str, Any]] = []
    warnings: List[Dict[str, Any]] = []

    def check(name, a_path, b_value, b_desc):
        a = _to_dec(get_by_path(PAYLOAD, a_path))
        b = _to_dec(b_value)
        if a is None or b is None:
            return
        d = _rel_diff(a, b)
        rec = {"check": name, "path": a_path, "value": float(a), "expected": float(b),
               "expected_from": b_desc, "rel_diff": round(d, 6)}
        if d > _ARITH_REL_TOL * 4:
            issues.append(rec)
        elif d > _ARITH_REL_TOL:
            warnings.append(rec)

    # 1) scope1 components sum to total, per year
    for i, row in enumerate(PAYLOAD.get("scope1") or []):
        gas, fleet, tot = _to_dec(row.get("scope1_gas_tco2e")), _to_dec(row.get("scope1_fleet_tco2e")), _to_dec(row.get("scope1_total_tco2e"))
        if None not in (gas, fleet, tot):
            check(f"scope1_components_sum_{row.get('reporting_year')}", f"scope1[{i}].scope1_total_tco2e",
                  gas + fleet, "scope1_gas_tco2e + scope1_fleet_tco2e")

    # 2) headline KPIs match the reporting-year rows of the period tables
    year_suffix = str(REPORTING_YEAR)
    kpi_map = {
        f"reporting_kpis.scope1_{year_suffix}_tco2e": ("scope1", "scope1_total_tco2e"),
        f"reporting_kpis.scope2_location_{year_suffix}_tco2e": ("scope2", "scope2_location_tco2e"),
        f"reporting_kpis.scope2_market_{year_suffix}_tco2e": ("scope2", "scope2_market_tco2e"),
        f"reporting_kpis.scope3_travel_{year_suffix}_tco2e": ("scope3_travel", "scope3_travel_tco2e"),
        f"reporting_kpis.financed_emissions_{year_suffix}_tco2e": ("financed_emissions", "financed_em_loans_tco2e"),
        f"reporting_kpis.total_assets_{year_suffix}_meur": ("financial_summary", "total_assets_meur"),
        f"reporting_kpis.total_loans_{year_suffix}_meur": ("financial_summary", "total_loans_meur"),
    }
    for kpi_path, (root, field) in kpi_map.items():
        for row in PAYLOAD.get(root) or []:
            if int(row.get("reporting_year") or -1) == REPORTING_YEAR:
                check(f"kpi_matches_{root}_{field}", kpi_path, row.get(field), f"{root}[year={REPORTING_YEAR}].{field}")

    # 3) financed-emissions lending intensity = financed / total_loans
    for i, row in enumerate(PAYLOAD.get("financed_emissions") or []):
        fe, loans, inten = (_to_dec(row.get("financed_em_loans_tco2e")), _to_dec(row.get("total_loans_meur")),
                            _to_dec(row.get("carbon_intensity_tco2e_per_meur_lending")))
        if None not in (fe, loans, inten) and loans:
            check(f"lending_intensity_{row.get('reporting_year')}",
                  f"financed_emissions[{i}].carbon_intensity_tco2e_per_meur_lending", fe / loans,
                  "financed_em_loans_tco2e / total_loans_meur")

    # 4) green-loan share = green_loans / total_loans, per year
    for i, row in enumerate(PAYLOAD.get("financial_summary") or []):
        g, t, pct = _to_dec(row.get("green_loans_meur")), _to_dec(row.get("total_loans_meur")), _to_dec(row.get("green_loans_pct"))
        if None not in (g, t, pct) and t:
            check(f"green_loans_pct_{row.get('reporting_year')}", f"financial_summary[{i}].green_loans_pct",
                  g / t * 100, "green_loans_meur / total_loans_meur * 100")

    # 5) percentage fields inside [0, 100]
    for path, value in FLAT_PAYLOAD.items():
        if re.search(r"(_pct|_percentage)$", path):
            d = _to_dec(value)
            if d is not None and not (Decimal(0) <= d <= Decimal(100)):
                issues.append({"check": "percentage_out_of_range", "path": path, "value": float(d)})

    # 6) target internal coherence: baseline_year < target_year; reduction pct in (0,100]
    for i, t in enumerate(PAYLOAD.get("targets") or []):
        by, ty = t.get("baseline_year"), t.get("target_year")
        if by is not None and ty is not None and int(by) >= int(ty):
            issues.append({"check": "target_years_inverted", "path": f"targets[{i}]", "baseline_year": by, "target_year": ty})
        red = _to_dec(t.get("target_value_pct_reduction"))
        if red is not None and not (Decimal(0) < red <= Decimal(100)):
            issues.append({"check": "target_reduction_out_of_range", "path": f"targets[{i}]", "value": float(red)})

    result = {"passed": len(issues) == 0, "hard_issues": issues, "warnings": warnings,
              "tolerance": _ARITH_REL_TOL, "checks_note":
              "Generic schema-discovering audit: component sums, KPI-vs-table equality, ratio identities, "
              "percentage bounds, target-year coherence."}
    write_json(result, STAGE_DIRS["audit"] / "payload_arithmetic_audit.json")
    return result

PAYLOAD_PROFILE = build_payload_profile()
write_json(PAYLOAD_PROFILE, STAGE_DIRS["audit"] / "payload_profile.json")
ARITH_AUDIT = run_payload_arithmetic_audit()
print("Payload profile written | arithmetic audit passed:", ARITH_AUDIT["passed"],
      f"| hard issues: {len(ARITH_AUDIT['hard_issues'])}, warnings: {len(ARITH_AUDIT['warnings'])}")
if not ARITH_AUDIT["passed"]:
    print("!! Hard arithmetic contradictions in payload — inspect 00_payload_audit before generating.")


## 5 · Requirement applicability (predicates + assessed conditionals)

In [ ]:
# ============================================================
# CELL 5 — REQUIREMENT APPLICABILITY ENGINE
# Your data constraint, made explicit: a requirement can only ever count
# against the report if it is APPLICABLE given the payload profile AND
# supported by verified evidence. Three-way decision, in priority order:
#   1. Deterministic predicates (profile-driven) for known conditionals.
#   2. LLM assessment (batched, logged rationale) for requirements whose
#      text carries conditional language not covered by a predicate.
#   3. Default: applicable.
# not_applicable requirements are excluded from planning, coverage and
# scoring, and recorded with their rationale in 01_applicability/.
# ============================================================
_CONDITIONAL_RE = re.compile(
    r"\bif\b|\bunless\b|where applicable|if applicable|need not|is permitted|may elect|"
    r"\bexempt\b|only if|to the extent|in the (?:first|initial) (?:annual )?reporting period|"
    r"commercially sensitive|impracticable", re.IGNORECASE)

def _pred(profile: Dict[str, Any]):
    """Named deterministic predicates: id -> (applicable: bool, rationale: str)."""
    P = profile
    return {
        "insurance_activities": (P["business_lines"]["insurance"],
                                 "Entity has no insurance business lines in the payload."),
        "asset_management_activities": (P["business_lines"]["asset_management"],
                                        "Asset-management exposure present in financed_emissions_equity/sovereign."),
        "commercial_banking_activities": (P["business_lines"]["commercial_banking"],
                                          "Commercial lending present in financed_emissions."),
        "carbon_credits_used": (P["has_carbon_credits"], "carbon_credits records present in payload."),
        "internal_carbon_price_used": (P["has_internal_carbon_price"], "internal_carbon_price records present."),
        "remuneration_linked": (P["has_remuneration_link"], "Executive remuneration ESG linkage present in governance."),
        "scope3_material": (P["has_scope3_categories"], "Included Scope 3 categories present."),
        "comparatives_available": (P["has_comparatives"], "Comparative years present in metadata."),
        "transition_plan_exists": (P["has_transition_plan"], "transition_plan.has_transition_plan is true."),
        "scenario_analysis_performed": (P["has_scenario_analysis"], "climate_scenarios records present."),
        "targets_set": (P["has_targets"], "targets records present."),
        "data_gaps_exist": (P["has_data_gaps"], "metadata.data_gaps present (estimation-uncertainty clauses apply)."),
        "first_reporting_period": (False, "Comparative periods exist, so first-annual-period reliefs do not apply."
                                   if P["has_comparatives"] else "Assumed first reporting period."),
    }

# tag/keyword → predicate routing for the deterministic layer
_PREDICATE_ROUTES = [
    (lambda r: "insurance" in (r.get("evidence_tags") or []), "insurance_activities"),
    (lambda r: "asset_management" in (r.get("evidence_tags") or []), "asset_management_activities"),
    (lambda r: "carbon_credits" in (r.get("evidence_tags") or []), "carbon_credits_used"),
    (lambda r: re.search(r"internal carbon price", r["requirement_text"], re.I), "internal_carbon_price_used"),
    (lambda r: "remuneration" in (r.get("evidence_tags") or []), "remuneration_linked"),
    (lambda r: re.search(r"first annual reporting period|transition relief", r["requirement_text"], re.I),
     "first_reporting_period"),
    (lambda r: re.search(r"comparative", r["requirement_text"], re.I), "comparatives_available"),
    (lambda r: re.search(r"transition plan", r["requirement_text"], re.I), "transition_plan_exists"),
    (lambda r: re.search(r"scenario analysis", r["requirement_text"], re.I) and
               re.search(r"\bif\b|where|unless", r["requirement_text"], re.I), "scenario_analysis_performed"),
]

def _mock_applicability(system: str, user: str) -> Dict[str, Any]:
    payload = json.loads(user[user.index("{"):]) if "{" in user else {}
    return {"assessments": [{"requirement_id": r["requirement_id"], "applicable": True,
                             "rationale": "mock: defaulted applicable"} for r in payload.get("requirements", [])]}
MOCK_HANDLERS["applicability_assessor"] = _mock_applicability

def assess_applicability() -> Dict[str, List[Dict[str, Any]]]:
    preds = _pred(PAYLOAD_PROFILE)
    out: Dict[str, List[Dict[str, Any]]] = {}
    for section in SECTIONS:
        records, needs_llm = [], []
        for r in requirements_by_section[section]:
            decided = None
            for match, pred_name in _PREDICATE_ROUTES:
                if match(r):
                    ok, why = preds[pred_name]
                    decided = {"requirement_id": r["requirement_id"], "applicable": bool(ok),
                               "decided_by": f"predicate:{pred_name}", "rationale": why}
                    break
            if decided is None:
                if _CONDITIONAL_RE.search(r["requirement_text"]):
                    needs_llm.append(r)
                else:
                    decided = {"requirement_id": r["requirement_id"], "applicable": True,
                               "decided_by": "default", "rationale": "Unconditional disclosure requirement."}
            if decided:
                records.append(decided)

        # batched LLM assessment for residual conditional-language requirements
        for i in range(0, len(needs_llm), 20):
            batch = needs_llm[i:i + 20]
            prompt = {
                "entity_profile": PAYLOAD_PROFILE,
                "requirements": [{"requirement_id": r["requirement_id"],
                                  "requirement_text": r["requirement_text"]} for r in batch],
            }
            try:
                obj = llm_json(
                "applicability_assessor:" + SECTION_SLUGS[section],
                "You determine whether conditional IFRS S1/S2 disclosure requirements apply to a specific "
                "entity, using ONLY the entity profile provided. Answer for each requirement. If the condition "
                "in the requirement text is not determinable from the profile, treat the requirement as "
                "applicable (conservative). Return JSON only: "
                '{"assessments":[{"requirement_id":"","applicable":true,"rationale":""}]}',
                json.dumps(prompt, ensure_ascii=False), max_tokens=3000)
                got = {a.get("requirement_id"): a for a in obj.get("assessments", [])}
                for r in batch:
                    a = got.get(r["requirement_id"], {})
                    records.append({"requirement_id": r["requirement_id"],
                                    "applicable": bool(a.get("applicable", True)),
                                    "decided_by": "llm_assessment",
                                    "rationale": str(a.get("rationale", "assessor returned no rationale; defaulted applicable"))[:400]})
            except Exception as exc:
                # Assessor failed for this batch: conservatively keep requirements
                # applicable, but RECORD that they were not truly assessed so the
                # human-review layer can see it rather than trusting a silent default.
                for r in batch:
                    records.append({"requirement_id": r["requirement_id"], "applicable": True,
                                    "decided_by": "llm_assessment_failed_defaulted_applicable",
                                    "rationale": f"Assessor error ({exc!r}); defaulted applicable pending review."[:400]})
        out[section] = records
        write_json(records, STAGE_DIRS["applicability"] / f"applicability_{SECTION_SLUGS[section]}.json")
    return out

APPLICABILITY = assess_applicability()
APPLICABLE_IDS: Dict[str, set] = {
    s: {a["requirement_id"] for a in APPLICABILITY[s] if a["applicable"]} for s in SECTIONS
}
for _s in SECTIONS:
    na = len(requirements_by_section[_s]) - len(APPLICABLE_IDS[_s])
    print(f"{_s}: {len(APPLICABLE_IDS[_s])} applicable, {na} not applicable")


## 6 · Evidence binding — lexical proposer, LLM verifier, cached

In [ ]:
# ============================================================
# CELL 6 — EVIDENCE BINDING: lexical proposer + LLM verifier (cached)
# The proposer is a cheap deterministic candidate generator (tag routing +
# keyword overlap). It decides NOTHING on its own: every proposed binding is
# confirmed or rejected by a batched LLM verifier constrained to real payload
# paths. Verified bindings are cached against a payload+requirements hash.
# Requirement status after this stage:
#   supported            — applicable, with >=1 verified binding
#   unsupported          — applicable, no verified binding (AUDIT ONLY;
#                          never penalizes the generated report)
#   not_applicable       — excluded upstream
# ============================================================
_STOPWORDS = set("the a an and or of to in for on by with as at from that which shall entity "
                 "disclose disclosure information about its their be is are was were this those "
                 "these an any each other such may can under over".split())

def _tokens(text: str) -> List[str]:
    return [t for t in re.findall(r"[a-z0-9_]+", str(text).lower()) if len(t) > 2 and t not in _STOPWORDS]

def _record_paths_for_section(section: str) -> List[Tuple[str, Any]]:
    """Candidate evidence units are RECORDS (list items / dict roots), not leaves —
    a whole governance[0] row is one evidence unit with all its fields."""
    items: List[Tuple[str, Any]] = []
    for root in SECTION_PAYLOAD_ROOTS[section]:
        val = PAYLOAD.get(root)
        if isinstance(val, list):
            for i, row in enumerate(val):
                items.append((f"{root}[{i}]", row))
        elif isinstance(val, dict):
            items.append((root, val))
    return items

def _propose_candidates(section: str, req: Dict[str, Any], record_items, max_candidates: int = 6):
    tags = req.get("evidence_tags") or []
    tag_roots = set()
    for t in tags:
        tag_roots.update(TAG_TO_ROOTS.get(t, []))
    req_kw = set(_tokens(req["requirement_text"]))
    scored = []
    for path, row in record_items:
        root = root_of_path(path)
        score = 0
        if tag_roots:
            score += 4 if root in tag_roots else -2
        row_kw = set(_tokens(json.dumps(row, default=str)[:4000]))
        score += len(req_kw & row_kw) // 3
        if score > 0:
            scored.append((score, path))
    # single-dict context roots (e.g. general_requirements_context, metadata, bank) are
    # always eligible: their fields collectively evidence preparation/boundary clauses.
    context_roots = {"general_requirements_context", "metadata", "bank"}
    for path, row in record_items:
        if root_of_path(path) in (context_roots & set(tag_roots or context_roots)) and path not in [p for _, p in scored]:
            scored.append((1, path))
    scored.sort(reverse=True)
    # dedupe by root, keep at most 2 records per root so multi-row roots don't flood
    seen_root = Counter()
    out = []
    for score, path in scored:
        r = root_of_path(path)
        if seen_root[r] >= 2:
            continue
        seen_root[r] += 1
        out.append(path)
        if len(out) >= max_candidates:
            break
    return out

def _record_preview(path: str, max_chars: int = 700) -> str:
    return json.dumps(get_by_path(PAYLOAD, path), ensure_ascii=False, default=str)[:max_chars]

def _mock_binding_verifier(system: str, user: str) -> Dict[str, Any]:
    payload = json.loads(user[user.index("{"):])
    verdicts = []
    for item in payload.get("items", []):
        cands = item.get("candidate_paths", [])
        verdicts.append({"requirement_id": item["requirement_id"],
                         "verified_paths": cands[:2],
                         "verdict": "supports" if cands else "no_evidence",
                         "note": "mock: accepted top candidates"})
    return {"verdicts": verdicts}
MOCK_HANDLERS["binding_verifier"] = _mock_binding_verifier

def build_verified_bindings() -> Dict[str, List[Dict[str, Any]]]:
    cache_key = sha12({"payload": sha12(PAYLOAD), "reqs": {s: [r["requirement_id"] for r in requirements_by_section[s]] for s in SECTIONS},
                       "roots": SECTION_PAYLOAD_ROOTS, "mode": LLM_MODE})
    cache_path = STAGE_DIRS["cache"] / f"bindings_{cache_key}.json"
    cached = read_json(cache_path)
    if cached:
        print(f"Evidence bindings loaded from cache ({cache_key}).")
        return cached

    result: Dict[str, List[Dict[str, Any]]] = {}
    for section in SECTIONS:
        record_items = _record_paths_for_section(section)
        rows = []
        applicable = [r for r in requirements_by_section[section] if r["requirement_id"] in APPLICABLE_IDS[section]]
        proposals = {r["requirement_id"]: _propose_candidates(section, r, record_items) for r in applicable}

        # batched verification
        BATCH = 20
        for i in range(0, len(applicable), BATCH):
            batch = applicable[i:i + BATCH]
            items = []
            for r in batch:
                cands = proposals[r["requirement_id"]]
                items.append({"requirement_id": r["requirement_id"],
                              "requirement_text": r["requirement_text"][:900],
                              "candidate_paths": cands,
                              "candidate_previews": {p: _record_preview(p) for p in cands}})
            obj = llm_json(
                "binding_verifier:" + SECTION_SLUGS[section],
                "You verify whether candidate payload records provide the underlying FACTS on which an "
                "IFRS S1/S2 disclosure would rest. A record SUPPORTS a requirement when it contains the "
                "data the disclosure is about (e.g. boundary, currency, comparative years, methodology, "
                "governance facts, metrics) — the payload need NOT contain a meta-statement that the entity "
                "'applies paragraph X'. Use 'partial' when only some elements are present, 'no_evidence' "
                "only when nothing in the candidates relates to the requirement. NEVER return a path not in "
                "candidate_paths. Return JSON only: "
                '{"verdicts":[{"requirement_id":"","verified_paths":[],"verdict":"supports|partial|no_evidence","note":""}]}',
                json.dumps({"items": items}, ensure_ascii=False), max_tokens=3500)
            got = {v.get("requirement_id"): v for v in obj.get("verdicts", [])}
            for r in batch:
                v = got.get(r["requirement_id"], {})
                allowed = set(proposals[r["requirement_id"]])
                verified = [p for p in (v.get("verified_paths") or []) if p in allowed]
                verdict = v.get("verdict", "no_evidence")
                status = "supported" if (verified and verdict in ("supports", "partial")) else "unsupported"
                rows.append({"requirement_id": r["requirement_id"], "section_name": section,
                             "status": status, "binding_strength": verdict if verified else "no_evidence",
                             "verified_paths": verified, "proposed_paths": proposals[r["requirement_id"]],
                             "verifier_note": str(v.get("note", ""))[:300]})
        result[section] = rows
        write_json(rows, STAGE_DIRS["bindings"] / f"bindings_{SECTION_SLUGS[section]}.json")

    write_json(result, cache_path)
    return result

BINDINGS = build_verified_bindings()
SUPPORTED: Dict[str, List[Dict[str, Any]]] = {}
UNSUPPORTED_REGISTER: Dict[str, List[Dict[str, Any]]] = {}
for _s in SECTIONS:
    SUPPORTED[_s] = [b for b in BINDINGS[_s] if b["status"] == "supported"]
    UNSUPPORTED_REGISTER[_s] = [
        {**b, "policy": "AUDIT ONLY. Applicable requirement without payload evidence. "
                        "Never penalizes the generated report; never mentioned in report prose."}
        for b in BINDINGS[_s] if b["status"] == "unsupported"]
    write_json(UNSUPPORTED_REGISTER[_s],
               STAGE_DIRS["bindings"] / f"unsupported_register_{SECTION_SLUGS[_s]}.json")
    print(f"{_s}: supported {len(SUPPORTED[_s])} | unsupported (audit-only) {len(UNSUPPORTED_REGISTER[_s])}")


## 7 · Fact table + section IR (numbers bound before writing)

In [ ]:
# ============================================================
# CELL 7 — FACT TABLE + SECTION INTERMEDIATE REPRESENTATION (IR)
# Facts are bound BEFORE writing. Every number the report may contain is a
# fact-table entry with path, typed value, unit, and period. The writer will
# reference numeric values ONLY as {{FACT:key}} tokens (Cell 9), which a
# deterministic renderer substitutes (Cell 10) — numeric grounding becomes
# checkable exactly, and multi-period values carry their year as data.
# ============================================================
_UNIT_RE = [
    (re.compile(r"_tco2e(_|$)"), "tCO2e"),
    (re.compile(r"_meur(_|$)"), "EUR million"),
    (re.compile(r"_eur_per_tco2e"), "EUR/tCO2e"),
    (re.compile(r"_pct(_|$)|_percentage"), "%"),
    (re.compile(r"tco2e_per_meur"), "tCO2e per EUR million"),
    (re.compile(r"_year$"), "year"),
]

def _unit_for(leaf: str) -> Optional[str]:
    for rx, unit in _UNIT_RE:
        if rx.search(leaf):
            return unit
    return None

def _format_value(v: Any, unit: Optional[str]) -> str:
    if isinstance(v, bool):
        return "yes" if v else "no"
    if unit == "year" and isinstance(v, (int, float)):
        return str(int(v))
    if isinstance(v, (int, float)):
        a = abs(float(v))
        if unit == "%":
            s = f"{float(v):.1f}"
        elif a >= 1000:
            s = f"{float(v):,.0f}"
        elif a >= 10:
            s = f"{float(v):,.1f}".rstrip("0").rstrip(".")
        else:
            s = f"{float(v):,.2f}".rstrip("0").rstrip(".")
        return s
    return str(v)

def _year_of_record(record: Any, path: str) -> Optional[int]:
    if isinstance(record, dict):
        for key in ("reporting_year", "vintage_year", "baseline_year"):
            y = record.get(key)
            if y is not None:
                try:
                    return int(y)
                except Exception:
                    return None
    m = re.search(r"_(19|20)(\d\d)_", path + "_")
    if m:
        return int(m.group(0).strip("_"))
    return None

def build_fact_table(section: str) -> Dict[str, Dict[str, Any]]:
    facts: Dict[str, Dict[str, Any]] = {}
    seen_paths = set()
    for b in SUPPORTED[section]:
        for rec_path in b["verified_paths"]:
            if rec_path in seen_paths:
                continue
            seen_paths.add(rec_path)
            record = get_by_path(PAYLOAD, rec_path)
            year = _year_of_record(record, rec_path)
            flat = flatten_json(record, rec_path) if isinstance(record, (dict, list)) else {rec_path: record}
            for leaf_path, value in flat.items():
                if value is None or (isinstance(value, str) and not value.strip()):
                    continue
                leaf = re.split(r"[.\[]", leaf_path)[-1].strip("]")
                if leaf in ("bank_id", "summary_id", "is_synthetic", "lei_code"):
                    continue
                unit = _unit_for(leaf)
                # numeric facts and short strings become facts; long strings stay as context
                is_bool = isinstance(value, bool)
                is_num = isinstance(value, (int, float)) and not is_bool
                # Booleans are preconditions, not disclosable values. They are recorded
                # as section context (Cell 9 states them in prose) but are NOT tokenisable
                # facts, so the writer can never emit "as indicated by yes".
                if is_bool:
                    continue
                key_year = _year_of_record(get_by_path(PAYLOAD, re.sub(r"\.[^.\[\]]+$", "", leaf_path)), leaf_path) or year
                key = re.sub(r"[^a-z0-9_.]", "_", leaf_path.lower())
                facts[key] = {
                    "fact_id": key, "path": leaf_path, "value": value,
                    "value_rendered": _format_value(value, unit),
                    "unit": unit, "period": key_year,
                    "kind": "number" if is_num else "text",
                }
    # deterministic DERIVED facts (formula logged) — counts and shares the writer commonly needs
    def add_derived(key, value, unit, period, formula):
        facts[key] = {"fact_id": key, "path": None, "value": value,
                      "value_rendered": _format_value(value, unit), "unit": unit,
                      "period": period, "kind": "number", "derived_from": formula}
    if section == "Governance" and PAYLOAD.get("board_minutes"):
        for y in ALL_YEARS:
            rows = [m for m in PAYLOAD["board_minutes"] if int(m.get("reporting_year") or -1) == y]
            if rows:
                climate = sum(1 for m in rows if m.get("climate_agenda_flag"))
                add_derived(f"derived.board_meetings_{y}", len(rows), None, y,
                            f"count(board_minutes where reporting_year={y})")
                add_derived(f"derived.board_meetings_climate_pct_{y}", round(100 * climate / len(rows), 1), "%", y,
                            f"share of board_minutes[{y}] with climate_agenda_flag")
    if section == "Metrics and Targets" and PAYLOAD.get("scope3_categories"):
        inc = PAYLOAD_PROFILE["scope3_included_categories"]
        add_derived("derived.scope3_included_category_count", len(inc), None, REPORTING_YEAR,
                    "count(distinct included scope3 category_number)")
    if section == "Risk Management" and PAYLOAD.get("physical_risk_exposures"):
        rows = PAYLOAD["physical_risk_exposures"]
        add_derived("derived.physical_risk_counterparties_assessed", len(rows), None, REPORTING_YEAR,
                    "count(physical_risk_exposures)")
        add_derived("derived.physical_risk_high_flag_count",
                    sum(1 for r in rows if r.get("high_risk_flag")), None, REPORTING_YEAR,
                    "count(physical_risk_exposures where high_risk_flag)")
    return facts

# ---- subsection planning: group supported requirements by dominant tag -----
_SECTION_SUBSECTION_ORDER = {
    "General Requirements": [("Basis of preparation and reporting entity", {"materiality", "source_guidance"}),
                             ("Reporting period, boundary and comparatives", {"connected_information"}),
                             ("Judgements, estimates and data quality", {"ghg_emissions", "metrics"}),
                             ("Assurance and connected information", {"connected_information", "financial_effects"})],
    "Governance": [("Board oversight of sustainability- and climate-related matters", {"governance_body"}),
                   ("Management's role, delegation and escalation", {"management_role"}),
                   ("Skills, remuneration linkage and monitoring", {"remuneration", "governance_body"})],
    "Strategy": [("Climate-related risks and opportunities", {"materiality", "business_model_value_chain"}),
                 ("Business model and value chain effects", {"business_model_value_chain"}),
                 ("Strategy, decision-making and transition plan", {"strategy_decision_making"}),
                 ("Financial effects: current and anticipated", {"financial_effects"}),
                 ("Climate resilience and scenario analysis", {"scenario_analysis"})],
    "Risk Management": [("Identification and assessment of climate-related risks", {"risk_process", "materiality"}),
                        ("Management, prioritisation and monitoring of risks", {"risk_process"}),
                        ("Integration with enterprise risk management", {"risk_process", "connected_information"})],
    "Metrics and Targets": [("GHG emissions: Scope 1 and Scope 2", {"scope_1", "scope_2", "ghg_emissions"}),
                            ("GHG emissions: Scope 3 and financed emissions", {"scope_3", "financed_emissions"}),
                            ("Methodology, consolidation and data quality", {"ghg_emissions", "metrics"}),
                            ("Cross-industry and entity-specific metrics", {"metrics", "commercial_banking"}),
                            ("Carbon credits and internal carbon pricing", {"carbon_credits"}),
                            ("Targets and progress", {"targets"})],
}

def _plan_subsections(section: str) -> List[Dict[str, Any]]:
    order = _SECTION_SUBSECTION_ORDER[section]
    assigned: Dict[str, List[str]] = {h: [] for h, _ in order}
    for b in SUPPORTED[section]:
        req = REQ_INDEX[b["requirement_id"]]
        tags = set(req.get("evidence_tags") or [])
        best, best_score = order[0][0], -1
        for heading, tagset in order:
            score = len(tags & tagset)
            if score > best_score:
                best, best_score = heading, score
        assigned[best].append(b["requirement_id"])
    return [{"heading": h, "requirement_ids": assigned[h]} for h, _ in order if assigned[h]]

def build_section_ir(section: str) -> Dict[str, Any]:
    facts = build_fact_table(section)
    subsections = _plan_subsections(section)
    binding_by_id = {b["requirement_id"]: b for b in SUPPORTED[section]}
    # table specs: one per multi-period numeric family that has >= 2 years
    families: Dict[str, Dict[int, str]] = defaultdict(dict)
    for k, f in facts.items():
        if f["kind"] == "number" and f["period"] in ALL_YEARS and f["path"]:
            leaf = re.split(r"[.\[]", f["path"])[-1].strip("]")
            families[f"{root_of_path(f['path'])}.{leaf}"][f["period"]] = k
    table_specs = []
    multi = {fam: y2k for fam, y2k in families.items() if len(y2k) >= 2}
    by_root: Dict[str, List[Tuple[str, Dict[int, str]]]] = defaultdict(list)
    for fam, y2k in sorted(multi.items()):
        by_root[fam.split(".")[0]].append((fam, y2k))
    for root, fams in by_root.items():
        table_specs.append({
            "table_id": f"tbl_{SECTION_SLUGS[section]}_{root}",
            "caption": f"{root.replace('_', ' ').title()} by reporting year",
            "columns": ["Metric"] + [str(y) for y in ALL_YEARS],
            "rows": [{"metric": fam.split('.', 1)[1].replace('_', ' '),
                      "fact_keys_by_year": {str(y): y2k[y] for y in ALL_YEARS if y in y2k}}
                     for fam, y2k in fams],
        })
    ir = {
        "section_name": section,
        "section_number": SECTION_NUMBERS[section],
        "entity_name": ENTITY_NAME,
        "reporting_year": REPORTING_YEAR,
        "comparative_years": COMPARATIVE_YEARS,
        "subsections": [
            {"heading": s["heading"],
             "requirements": [
                 {"requirement_id": rid,
                  "requirement_text": REQ_INDEX[rid]["requirement_text"][:900],
                  "standard": REQ_INDEX[rid]["standard"],
                  "paragraph_id": REQ_INDEX[rid]["paragraph_id"],
                  "verified_paths": binding_by_id[rid]["verified_paths"],
                  "binding_strength": binding_by_id[rid]["binding_strength"]}
                 for rid in s["requirement_ids"]]}
            for s in subsections],
        "fact_table": facts,
        "table_specs": table_specs,
        "prohibitions": [{"requirement_id": p["requirement_id"], "text": p["requirement_text"][:500]}
                         for p in prohibitions_by_section[section]],
        "figure_specs": [],
    }
    if section == "Governance":
        ir["figure_specs"].append({
            "figure_id": "fig_governance_structure",
            "title": "Sustainability and climate governance structure",
            "spec": "Diagram: Board -> ESG Committee -> management-level committees; "
                    "connectors labelled 'informs' / 'reports to' / 'approves'. Rendered downstream.",
        })
    write_json(ir, STAGE_DIRS["ir"] / f"ir_{SECTION_SLUGS[section]}.json")
    return ir

SECTION_IR: Dict[str, Dict[str, Any]] = {s: build_section_ir(s) for s in SECTIONS}
for _s in SECTIONS:
    _ir = SECTION_IR[_s]
    print(f"{_s}: {len(_ir['subsections'])} subsections | {len(_ir['fact_table'])} facts | "
          f"{len(_ir['table_specs'])} table specs")


## 8 · Style contract (built-in; extracted style only via ablation arms) + firewall

In [ ]:
# ============================================================
# CELL 8 — STYLE CONTRACT, NO-COPY FIREWALL, LIMITATION POLICY
# Per sign-off: the extracted style system is NOT injected (STYLE_ARM="none").
# The built-in contract below is minimal, achievable, and evidence-compatible.
# STYLE_ARM="raw"/"refined" load the extracted style for the ablation harness.
# The SAME contract feeds the writer prompt, the deterministic style checks,
# and the style judge — one source of truth.
# ============================================================
BUILTIN_STYLE_CONTRACT = {
    "voice": "First-person plural for the entity's actions ('we disclose', 'the Group assesses'); "
             "third person / passive for externally defined IFRS requirements.",
    "tone": "Formal, neutral, precise, audit-ready. Process- and control-focused. No promotional, "
            "celebratory or advocacy language. No absolute assurances ('guarantees', 'ensures outcomes').",
    "structure": [
        "Numbered heading hierarchy: '## N.M Heading' for subsections and '### N.M.K Heading' for "
        "sub-subsections, where N is the section number provided in the context.",
        "Open each subsection with a one-sentence purpose statement before details.",
        "Paragraphs are single-idea, 2-5 sentences.",
        "Introduce every table with a lead-in sentence; every table has a caption line directly above it "
        "formatted exactly as 'Table: <caption>' (numbering is added at assembly).",
        "Close each subsection with one connectivity sentence linking to a related section of this report "
        "where the evidence supports the connection.",
    ],
    "sentences": [
        "Declarative subject-verb-object sentences, 18-34 words preferred.",
        "Define acronyms at first use (e.g., 'greenhouse gas (GHG)').",
        "Cautious modality for forward-looking statements ('is expected to', 'is intended to', 'may').",
    ],
    "numbers": [
        "Every numeric value MUST be written as a fact token {{FACT:key}} from the provided fact table — "
        "never as a literal number. Calendar years, heading numbers and list markers are the only exceptions.",
        "When a metric exists for several periods, either present the periods in a table or name the year "
        "in the same sentence as the token.",
    ],
    "figures": "Where a figure_spec is provided, emit a fenced block ```figure ... ``` containing the spec's "
               "title and description at the planned position; do not attempt ASCII art.",
    "prohibited": [
        "Missing-data or pipeline language: never mention missing data, unavailable information, payloads, "
        "synthetic data, placeholders, datasets, or internal processes — EXCEPT the sanctioned IFRS "
        "limitation statements listed under 'sanctioned_limitations'.",
        "No invented facts, entities, dates, commitments, frameworks or methodologies.",
    ],
    "sanctioned_limitations": [
        "estimation and measurement uncertainty arising from data gaps or proxy data, described qualitatively",
        "Scope 3 category inclusion/exclusion basis, including exclusion reasons recorded in evidence",
        "the level and scope of external assurance obtained",
        "use of proxy or estimated data in financed-emissions measurement (e.g., PCAF data quality)",
    ],
}

def load_style_contract() -> Dict[str, Any]:
    if STYLE_ARM == "none":
        return BUILTIN_STYLE_CONTRACT
    # ablation arms: overlay extracted style on the built-in contract
    global_style = read_json(STYLE_SYSTEM_DIR / "authoring" / "global_style_guide.json", default={})
    if STYLE_ARM == "raw":
        return {**BUILTIN_STYLE_CONTRACT, "extracted_global_style_raw": global_style}
    if STYLE_ARM == "refined":
        refined = dict(global_style)
        # refinement per audit §4.2: drop rules that contradict evidence grounding
        for key in ("what_to_avoid",):
            refined.pop(key, None)
        return {**BUILTIN_STYLE_CONTRACT, "extracted_global_style_refined": refined}
    raise ValueError(f"Unknown STYLE_ARM: {STYLE_ARM}")

STYLE_CONTRACT = load_style_contract()

# ---- no-copy firewall (safety, always on, independent of style arm) --------
FORBIDDEN_TERMS = sorted(set(
    (read_json(STYLE_SYSTEM_DIR / "authoring" / "language_rules" / "forbidden_reference_terms.json", default=[]) or [])
    + ["Emirates NBD", "Emirates NBD Group", "DenizBank", "Emirates Islamic", "Dubai", "UAE", "AED",
       "CBUAE", "Sustainalytics", "KPMG", "Microsoft Sustainability Manager"]))

# ---- pipeline-language cleanliness -----------------------------------------
CLEANLINESS_PATTERNS = [
    r"\bpayload\b", r"\bsynthetic\b", r"\bplaceholder\b", r"\bdataset\b", r"\bpipeline\b",
    r"\bTODO\b", r"\bTBD\b", r"\[insert", r"as evidenced by", r"\bmock\b", r"\bdebug\b",
    r"human review", r"audit[- ]only", r"not (?:available|provided) in", r"\bJSON\b", r"\bregister(?:ed)? in the data\b",
]

# Sanctioned limitation sentences are recognised (and allowed) by these markers:
SANCTIONED_LIMITATION_MARKERS = [
    r"estimation (?:and measurement )?uncertaint", r"proxy", r"PCAF", r"data quality",
    r"limited assurance", r"reasonable assurance", r"assurance", r"excluded? (?:from Scope 3|categor)",
]
print(f"Style contract loaded (arm={STYLE_ARM}) | firewall terms: {len(FORBIDDEN_TERMS)}")


## 9 · Writer — numeric values only as `{{FACT:key}}` tokens

In [ ]:
# ============================================================
# CELL 9 — SECTION WRITER (fact-token generation)
# The writer sees: section IR (subsections + requirements + fact table +
# table/figure specs), the style contract, and the prohibitions. It must
# express every number as {{FACT:key}}. A deterministic mock writer renders
# the same IR offline so the whole verification chain is testable.
# ============================================================
def _writer_prompt(section: str) -> Tuple[str, str]:
    ir = SECTION_IR[section]
    system = (
        "You are a senior IFRS S1/S2 sustainability disclosure writer. You write final-report Markdown "
        "strictly from the provided fact table and evidence. HARD RULES: (1) every numeric value is a "
        "{{FACT:key}} token whose key exists in fact_table — never a literal number except calendar years, "
        "heading numbers and list markers; (2) never invent facts, names, dates or commitments; "
        "(3) address every requirement listed in each subsection using its verified evidence; "
        "(3a) never write a token followed by a bare boolean word; state existence in prose "
        "(e.g. 'CEO remuneration is linked to ESG performance') rather than 'linked as indicated by yes'; "
        "(3b) when a fact carries unit '%', write the value with a percent sign; "
        "(4) obey the style contract and the prohibitions; (5) no missing-data language except the "
        "sanctioned limitation statements. Return JSON only: "
        '{"section_name":"","draft_markdown":"","writer_notes":""}'
    )
    user = json.dumps({
        "task": f"Write section {ir['section_number']} — {section} — for {ENTITY_NAME}'s IFRS S1/S2 report, "
                f"reporting year {REPORTING_YEAR}.",
        "style_contract": STYLE_CONTRACT,
        "prohibitions": ir["prohibitions"],
        "section_ir": {k: ir[k] for k in ("section_number", "subsections", "table_specs", "figure_specs")},
        "fact_table": {k: {kk: f[kk] for kk in ("value_rendered", "unit", "period", "kind")}
                       for k, f in ir["fact_table"].items()},
    }, ensure_ascii=False)
    return system, user

def _render_table_from_spec(spec: Dict[str, Any]) -> str:
    lines = [f"Table: {spec['caption']}", "", "| " + " | ".join(spec["columns"]) + " |",
             "|" + "|".join(["---"] * len(spec["columns"])) + "|"]
    for row in spec["rows"]:
        cells = [row["metric"]]
        for col in spec["columns"][1:]:
            k = row["fact_keys_by_year"].get(col)
            cells.append("{{FACT:%s}}" % k if k else "—")
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)

def _mock_writer(system: str, user: str) -> Dict[str, Any]:
    ctx = json.loads(user)
    ir_ctx = ctx["section_ir"]
    facts = ctx["fact_table"]
    n = ir_ctx["section_number"]
    section = ctx["task"].split("—")[1].strip()
    out = []
    fact_keys = list(facts.keys())
    used = 0
    for m, sub in enumerate(ir_ctx["subsections"], start=1):
        out.append(f"## {n}.{m} {sub['heading']}")
        out.append(f"This subsection explains how we address {sub['heading'].lower()} in accordance with "
                   f"IFRS S1 and IFRS S2.")
        # one grounded sentence per requirement using a fact token where possible
        for req in sub["requirements"][:6]:
            key = None
            for cand in fact_keys[used:used + 40]:
                if facts[cand]["kind"] == "number":
                    key = cand
                    break
            if key:
                used = fact_keys.index(key) + 1
                yr = facts[key]["period"] or ctx["task"][-5:-1]
                out.append(f"In {yr}, the relevant measure was {{{{FACT:{key}}}}}"
                           f"{(' ' + facts[key]['unit']) if facts[key]['unit'] else ''}, which we monitor "
                           f"as part of our response to {req['standard']} paragraph {req['paragraph_id']}.")
            else:
                out.append(f"We describe our approach responsive to {req['standard']} paragraph "
                           f"{req['paragraph_id']} using the verified governance and process evidence.")
        out.append("This connects to the related disclosures elsewhere in this report.")
        out.append("")
    for spec in ir_ctx.get("table_specs", [])[:2]:
        out.append("The following table presents the supporting measures by reporting year.")
        out.append(_render_table_from_spec(spec))
        out.append("")
    for fig in ir_ctx.get("figure_specs", []):
        out.append("```figure\n" + fig["title"] + "\n" + fig["spec"] + "\n```")
    return {"section_name": section, "draft_markdown": "\n\n".join(out), "writer_notes": "mock writer"}
MOCK_HANDLERS["writer"] = _mock_writer
def _mock_patch_writer(system: str, user: str) -> Dict[str, Any]:
    ctx = json.loads(user)
    req = ctx.get("requirement", {})
    return {"patch_markdown":
            f"We additionally disclose the matters required by {req.get('standard', 'IFRS S1')} "
            f"paragraph {req.get('paragraph_id', '')} using the verified evidence for this requirement.",
            "notes": "mock patch"}
MOCK_HANDLERS["patch_writer"] = _mock_patch_writer

def write_section_draft(section: str) -> Dict[str, Any]:
    system, user = _writer_prompt(section)
    obj = llm_json("writer:" + SECTION_SLUGS[section], system, user, max_tokens=9000, temperature=0.1)
    obj.setdefault("draft_markdown", "")
    return obj
print("Writer ready.")


## 10 · Deterministic rendering & verification gates

In [ ]:
# ============================================================
# CELL 10 — DETERMINISTIC RENDERING + VERIFICATION GATES
# Order: substitute fact tokens with sentinel markers -> scan the remaining
# text for literal numbers (hard failure except years/headings/list markers)
# -> check period consistency of every token -> style mechanics -> firewall
# -> cleanliness -> prohibitions lexicon. Gates run on the EXACT text that
# will be stored; the renderer output and gate input can never diverge.
# ============================================================
_TOKEN_RE = re.compile(r"\{\{FACT:([a-z0-9_.\[\]]+)\}\}", re.IGNORECASE)
_SENT_OPEN, _SENT_CLOSE = "\uE000", "\uE001"

def gate_token_substitution(section: str, unknown: List[str]) -> Dict[str, Any]:
    return {"gate": "fact_token_substitution", "passed": not unknown,
            "failures": [{"type": "unknown_fact_key", "key": k} for k in unknown]}

_ALLOWED_NUMBER_RE = re.compile(r"^(19|20)\d\d$")   # calendar years
# References to the standards themselves are structure, not data values:
_STRUCTURAL_REF_RE = re.compile(
    r"IFRS\s+S[12]\b|\bS[12]\b|\bparagraphs?\s+[A-Z]?\d+[A-Z]?(?:\([a-z]\))?|\bScope\s+[123]\b|"
    r"\bSection\s+\d+\b|"
    r"\bCategory\s+\d+\b|\bISAE\s*\d+\b", re.IGNORECASE)
_HEADING_NUM_RE = re.compile(r"^#{1,4}\s+\d+(?:\.\d+)*\s")
_LIST_MARKER_RE = re.compile(r"^\s*(?:\d+\.|[-*+])\s")

def gate_literal_numbers(section: str, rendered: str) -> Dict[str, Any]:
    failures = []
    for line in rendered.splitlines():
        scan_line = line
        if _HEADING_NUM_RE.match(line):
            scan_line = _HEADING_NUM_RE.sub("# ", line)
        if _LIST_MARKER_RE.match(scan_line):
            scan_line = _LIST_MARKER_RE.sub(" ", scan_line)
        clean = re.sub(f"{_SENT_OPEN}[^{_SENT_CLOSE}]*{_SENT_CLOSE}", " ", scan_line)
        clean = re.sub(r"Table:\s.*$", " ", clean)   # captions numbered only at assembly
        clean = _STRUCTURAL_REF_RE.sub(" ", clean)    # standard citations are not data
        clean = _TOKEN_RE.sub(" ", clean)             # unsubstituted tokens: reported by the token gate, not here
        # standalone numbers only: digits embedded in words (tCO2e, CET1, Tier1, 29B) are labels
        for tok in re.findall(r"(?<![A-Za-z0-9])\d[\d,]*\.?\d*%?(?![A-Za-z0-9])", clean):
            core = tok.rstrip("%").replace(",", "")
            if _ALLOWED_NUMBER_RE.match(core):
                continue
            failures.append({"type": "literal_number_outside_fact_table", "value": tok,
                             "line": line.strip()[:160]})
    return {"gate": "literal_numbers", "passed": not failures, "failures": failures[:60]}

def _substitute_with_keys(section: str, draft: str) -> Tuple[str, List[Dict[str, Any]], List[str]]:
    """Substitution that records, per token, the final line number and column context."""
    facts = SECTION_IR[section]["fact_table"]
    unknown, used = [], []
    out_lines = []
    table_header_years: set = set()
    for line in draft.splitlines():
        if line.strip().startswith("|") and re.search(r"(?:19|20)\d\d", line) and "---" not in line:
            table_header_years = set(int(y) for y in re.findall(r"(?:19|20)\d\d", line))
        elif not line.strip().startswith("|"):
            if line.strip():
                table_header_years = table_header_years if line.strip().startswith("Table:") else set()
        def _sub(m):
            key = m.group(1).lower()
            f = facts.get(key)
            if not f:
                unknown.append(key)
                return m.group(0)
            used.append({"key": key, "period": f["period"], "line_text": line,
                         "table_header_years": sorted(table_header_years)})
            return f"{_SENT_OPEN}{f['value_rendered']}{_SENT_CLOSE}"
        out_lines.append(_TOKEN_RE.sub(_sub, line))
    return "\n".join(out_lines), used, unknown

def gate_period_consistency(section: str, used: List[Dict[str, Any]]) -> Dict[str, Any]:
    facts = SECTION_IR[section]["fact_table"]
    fam_years: Dict[str, set] = defaultdict(set)
    for k, f in facts.items():
        if f["kind"] == "number" and f["period"]:
            leaf = re.split(r"[.\[]", (f["path"] or k))[-1].strip("]")
            fam_years[leaf].add(f["period"])
    failures, warnings = [], []
    for u in used:
        f = facts.get(u["key"]) or {}
        period = f.get("period")
        if not period:
            continue
        line_years = set(int(y) for y in re.findall(r"(?:19|20)\d\d", u["line_text"]))
        leaf = re.split(r"[.\[]", (f.get("path") or u["key"]))[-1].strip("]")
        multi = len(fam_years.get(leaf, set())) > 1
        in_table = u["line_text"].strip().startswith("|")
        ctx_years = set(u.get("table_header_years") or []) if in_table else line_years
        if ctx_years and period not in ctx_years and line_years and period not in line_years:
            failures.append({"type": "value_on_line_naming_different_year", "key": u["key"],
                             "fact_period": period, "line_years": sorted(line_years),
                             "line": u["line_text"].strip()[:160]})
        elif multi and not ctx_years and not line_years:
            warnings.append({"type": "multi_period_metric_without_year_context", "key": u["key"],
                             "fact_period": period, "line": u["line_text"].strip()[:160]})
    return {"gate": "period_consistency", "passed": not failures,
            "failures": failures[:40], "warnings": warnings[:40]}

def gate_style_mechanics(section: str, rendered: str) -> Dict[str, Any]:
    n = SECTION_NUMBERS[section]
    failures, warnings = [], []
    headings = re.findall(r"^(#{2,4})\s+(.*)$", rendered, flags=re.M)
    if not headings:
        failures.append({"type": "no_subsection_headings"})
    for hashes, text in headings:
        if not re.match(rf"^{n}(?:\.\d+)+\s", text):
            failures.append({"type": "heading_not_numbered", "heading": text[:100],
                             "expected_prefix": f"{n}.x"})
    lines = rendered.splitlines()
    for i, l in enumerate(lines):
        if l.strip().startswith("|") and (i == 0 or not lines[i-1].strip().startswith("|")):
            back = [x.strip() for x in lines[max(0, i-3):i] if x.strip()]
            # caption may sit on its own line OR be appended to the lead-in sentence
            has_caption = any("Table:" in b for b in back)
            if not has_caption:
                failures.append({"type": "table_without_caption", "near": l.strip()[:80]})
    # paragraph length (warning only)
    for para in re.split(r"\n\s*\n", rendered):
        if para.strip().startswith(("#", "|", "```", "Table:")):
            continue
        sents = re.split(r"(?<=[.!?])\s+", para.strip())
        if len(sents) >= 8:
            warnings.append({"type": "paragraph_too_long", "sentences": len(sents),
                             "start": para.strip()[:80]})
    return {"gate": "style_mechanics", "passed": not failures,
            "failures": failures[:40], "warnings": warnings[:40]}

def gate_firewall(rendered: str) -> Dict[str, Any]:
    hits = [t for t in FORBIDDEN_TERMS if re.search(rf"\b{re.escape(t)}\b", rendered)]
    return {"gate": "no_copy_firewall", "passed": not hits,
            "failures": [{"type": "forbidden_reference_term", "term": t} for t in hits]}

def gate_cleanliness(rendered: str) -> Dict[str, Any]:
    failures = []
    for pat in CLEANLINESS_PATTERNS:
        for m in re.finditer(pat, rendered, flags=re.IGNORECASE):
            line = rendered[:m.start()].rsplit("\n", 1)[-1] + rendered[m.start():].split("\n", 1)[0]
            if any(re.search(sm, line, re.IGNORECASE) for sm in SANCTIONED_LIMITATION_MARKERS):
                continue
            failures.append({"type": "pipeline_language", "pattern": pat, "line": line.strip()[:160]})
    return {"gate": "cleanliness", "passed": not failures, "failures": failures[:40]}

def finalize_rendered(rendered: str) -> str:
    """Strip sentinels; light whitespace normalization. This is the ONLY
    transformation applied after gating, and it is character-preserving for
    everything except the sentinel characters themselves."""
    out = rendered.replace(_SENT_OPEN, "").replace(_SENT_CLOSE, "")
    out = re.sub(r"[ \t]+\n", "\n", out)
    return re.sub(r"\n{3,}", "\n\n", out).strip() + "\n"

def run_deterministic_gates(section: str, draft: str) -> Dict[str, Any]:
    rendered, used, unknown = _substitute_with_keys(section, draft)
    gates = [
        gate_token_substitution(section, unknown),
        gate_literal_numbers(section, rendered),
        gate_period_consistency(section, used),
        gate_style_mechanics(section, rendered),
        gate_firewall(rendered),
        gate_cleanliness(rendered),
    ]
    return {"section_name": section, "passed": all(g["passed"] for g in gates),
            "gates": gates, "rendered_with_sentinels": rendered,
            "final_text": finalize_rendered(rendered),
            "facts_used": used, "fact_use_count": len(used)}
print("Deterministic verification ready.")


## 11 · LLM verification — output coverage (quote-checked), claims, style

In [ ]:
# ============================================================
# CELL 11 — LLM VERIFICATION (output-side)
# 1) OUTPUT-COVERAGE VERIFIER: for every SUPPORTED requirement, does the
#    final text actually disclose it? Verdicts carry a verbatim quote that
#    is deterministically checked against the text (whitespace-normalised);
#    a verdict whose quote does not appear is downgraded to 'unverified'.
# 2) QUALITATIVE-CLAIMS CHECK: non-numeric assertions (numbers are already
#    grounded by construction) are extracted and checked against the
#    verified evidence records; unsupported claims BLOCK.
# 3) STYLE JUDGE: grades ONLY the non-mechanical style rules, against the
#    same contract the writer saw, on a 0-10 scale.
# ============================================================
def _norm_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip().lower()

def _mock_coverage(system: str, user: str) -> Dict[str, Any]:
    payload = json.loads(user[user.index("{"):])
    text = payload["final_text"]
    verdicts = []
    for r in payload["requirements"]:
        pid = r["paragraph_id"]
        hit = re.search(rf"paragraph {re.escape(str(pid))}\b[^.\n]*", text)
        if hit:
            verdicts.append({"requirement_id": r["requirement_id"], "status": "addressed",
                             "quote": hit.group(0)[:120], "gap": ""})
        else:
            verdicts.append({"requirement_id": r["requirement_id"], "status": "missing",
                             "quote": "", "gap": "mock: no sentence references this clause"})
    return {"verdicts": verdicts}
MOCK_HANDLERS["coverage_verifier"] = _mock_coverage
MOCK_HANDLERS["claims_checker"] = lambda s, u: {"claims": []}
MOCK_HANDLERS["style_judge"] = lambda s, u: {"style_score_0_to_10": 8.0, "issues": [], "summary": "mock"}

def verify_output_coverage(section: str, final_text: str) -> List[Dict[str, Any]]:
    supported = SUPPORTED[section]
    results: List[Dict[str, Any]] = []
    BATCH = 15
    norm_text = _norm_ws(final_text)
    for i in range(0, len(supported), BATCH):
        batch = supported[i:i + BATCH]
        prompt = {
            "final_text": final_text,
            "requirements": [{"requirement_id": b["requirement_id"],
                              "paragraph_id": REQ_INDEX[b["requirement_id"]]["paragraph_id"],
                              "standard": REQ_INDEX[b["requirement_id"]]["standard"],
                              "requirement_text": REQ_INDEX[b["requirement_id"]]["requirement_text"][:900]}
                             for b in batch],
        }
        obj = llm_json(
            "coverage_verifier:" + SECTION_SLUGS[section],
            "You verify whether a report section actually discloses each listed IFRS S1/S2 requirement. "
            "For each requirement return status 'addressed' (the specific disclosure the clause demands is "
            "present), 'partial' (some elements present), or 'missing'. Include a verbatim quote (<=25 words) "
            "copied EXACTLY from the text that best evidences your verdict (empty for 'missing'), and for "
            "partial/missing a one-sentence gap description of what is absent. Judge only against the text. "
            'Return JSON only: {"verdicts":[{"requirement_id":"","status":"","quote":"","gap":""}]}',
            json.dumps(prompt, ensure_ascii=False), max_tokens=3800)
        got = {v.get("requirement_id"): v for v in obj.get("verdicts", [])}
        for b in batch:
            v = got.get(b["requirement_id"], {"status": "missing", "quote": "", "gap": "verifier returned no verdict"})
            status = v.get("status", "missing")
            quote = str(v.get("quote", ""))
            quote_ok = bool(quote) and _norm_ws(quote) in norm_text
            if status in ("addressed", "partial") and not quote_ok:
                status = "unverified"
            results.append({"requirement_id": b["requirement_id"], "status": status,
                            "quote": quote[:200], "quote_verified": quote_ok,
                            "gap": str(v.get("gap", ""))[:300]})
    return results

def check_qualitative_claims(section: str, final_text: str) -> Dict[str, Any]:
    # Evidence horizon = everything the writer could legitimately draw on for this
    # section (its payload roots), not only the narrowly verified per-requirement
    # bindings. A claim grounded in a section root the writer was given must not be
    # reported as unsupported just because it fell outside a single requirement's
    # binding. Records are compacted to keep the prompt bounded.
    evidence_records = {}
    for root in SECTION_PAYLOAD_ROOTS[section]:
        val = PAYLOAD.get(root)
        if isinstance(val, list):
            for i, row in enumerate(val[:25]):
                evidence_records[f"{root}[{i}]"] = row
        elif val is not None:
            evidence_records[root] = val
    obj = llm_json(
        "claims_checker:" + SECTION_SLUGS[section],
        "You audit a report section for QUALITATIVE factual assertions (entities, committee names, "
        "frameworks, methodologies, processes, commitments, dates as facts). Numeric values are grounded "
        "elsewhere — ignore them. Extract each material qualitative claim and decide from the evidence "
        "records whether it is supported. Return JSON only: "
        '{"claims":[{"claim_text":"","supported":true,"evidence_path":"","reason":""}]}',
        json.dumps({"final_text": final_text,
                    "evidence_records": {k: json.dumps(v, default=str)[:800] for k, v in evidence_records.items()}},
                   ensure_ascii=False), max_tokens=3800)
    claims = obj.get("claims", [])
    unsupported = [c for c in claims if c.get("supported") is False]
    return {"claims": claims, "unsupported": unsupported, "passed": not unsupported}

def judge_style(section: str, final_text: str) -> Dict[str, Any]:
    obj = llm_json(
        "style_judge:" + SECTION_SLUGS[section],
        "You judge a report section ONLY against the non-mechanical rules of the style contract provided "
        "(voice, tone, purpose statements, cautious modality, connectivity, prohibited promotional or "
        "absolute language). Mechanical rules (heading numbering, captions, paragraph length) are checked "
        "elsewhere — do not grade them. Score 0-10. Return JSON only: "
        '{"style_score_0_to_10":0,"issues":[{"rule":"","quote":"","fix":""}],"summary":""}',
        json.dumps({"style_contract": STYLE_CONTRACT, "final_text": final_text}, ensure_ascii=False),
        max_tokens=2500)
    obj["style_score_0_to_10"] = float(obj.get("style_score_0_to_10", 0.0))
    return obj

def run_llm_verification(section: str, final_text: str) -> Dict[str, Any]:
    coverage = verify_output_coverage(section, final_text)
    claims = check_qualitative_claims(section, final_text)
    style = judge_style(section, final_text)
    counts = Counter(v["status"] for v in coverage)
    return {"section_name": section, "coverage_verdicts": coverage,
            "coverage_counts": dict(counts), "claims_check": claims, "style_judge": style}
print("LLM verification ready.")


## 12 · Targeted repair — requirement patches & sentence-level fixes

In [ ]:
# ============================================================
# CELL 12 — TARGETED REPAIR (deterministic routing, span-level patches)
# The router never rewrites a whole section for a local defect:
#   unknown token / literal number / wrong-year line  -> sentence-level fix
#   missing or unverified requirement                 -> patch writer adds ONLY
#      that disclosure, inserted at its planned subsection
#   unsupported qualitative claim                     -> sentence removal/rewrite
# Whole-section redraft happens only if the draft has no valid structure at all.
# ============================================================
def _caption_bare_tables(draft: str) -> str:
    """Insert 'Table: <inferred caption>' above any table lacking one. Caption is
    derived from the table's own header row so it stays evidence-faithful."""
    lines = draft.splitlines()
    out = []
    i = 0
    while i < len(lines):
        l = lines[i]
        is_table_start = l.strip().startswith("|") and (i == 0 or not lines[i-1].strip().startswith("|"))
        if is_table_start:
            back = [x.strip() for x in out[-3:] if x.strip()]
            if not any("Table:" in b for b in back):
                headers = [h.strip() for h in l.strip().strip("|").split("|")]
                metric = headers[0] if headers else "Metric"
                out.append(f"Table: {metric} by reporting year" if any(re.search(r"20\d\d", x) for x in lines[i:i+2])
                           else f"Table: {metric}")
        out.append(l)
        i += 1
    return "\n".join(out)

def _subsection_of_requirement(section: str, requirement_id: str) -> Optional[str]:
    for sub in SECTION_IR[section]["subsections"]:
        if any(r["requirement_id"] == requirement_id for r in sub["requirements"]):
            return sub["heading"]
    return None

def _insert_after_heading(draft: str, heading: str, patch_md: str) -> str:
    lines = draft.splitlines()
    out, inserted = [], False
    i = 0
    while i < len(lines):
        out.append(lines[i])
        if not inserted and lines[i].lstrip("# ").strip().endswith(heading):
            # insert after this subsection's existing content: find next heading
            j = i + 1
            while j < len(lines) and not lines[j].startswith("## "):
                out.append(lines[j]); j += 1
            out.append(""); out.append(patch_md.strip()); out.append("")
            inserted = True
            i = j
            continue
        i += 1
    if not inserted:
        out += ["", patch_md.strip(), ""]
    return "\n".join(out)

def patch_missing_requirement(section: str, draft: str, requirement_id: str, gap: str) -> str:
    req = REQ_INDEX[requirement_id]
    binding = next(b for b in SUPPORTED[section] if b["requirement_id"] == requirement_id)
    facts = SECTION_IR[section]["fact_table"]
    relevant_fact_keys = [k for k, f in facts.items()
                          if f.get("path") and any(f["path"].startswith(p) for p in binding["verified_paths"])]
    prompt = {
        "requirement": {"requirement_id": requirement_id, "standard": req["standard"],
                        "paragraph_id": req["paragraph_id"], "requirement_text": req["requirement_text"]},
        "identified_gap": gap,
        "evidence_records": {p: json.dumps(get_by_path(PAYLOAD, p), default=str)[:900]
                             for p in binding["verified_paths"]},
        "fact_keys_available": {k: {kk: facts[k][kk] for kk in ("value_rendered", "unit", "period")}
                                for k in relevant_fact_keys[:40]},
        "style_contract": STYLE_CONTRACT,
    }
    obj = llm_json(
        "patch_writer:" + requirement_id,
        "You write ONLY the paragraph(s) (1-3) that disclose the single IFRS requirement provided, closing "
        "the identified gap, using only the evidence records. Numeric values must be {{FACT:key}} tokens "
        "from fact_keys_available. No headings. No missing-data language. Return JSON only: "
        '{"patch_markdown":"","notes":""}',
        json.dumps(prompt, ensure_ascii=False), max_tokens=1500, temperature=0.1)
    patch = str(obj.get("patch_markdown", "")).strip()
    if not patch:
        return draft
    heading = _subsection_of_requirement(section, requirement_id) or ""
    return _insert_after_heading(draft, heading, patch)

def _remove_sentence_containing(draft: str, needle: str) -> str:
    """Remove the first whole sentence (or table row) containing needle.
    Never touches heading lines, captions, fenced blocks or {{FACT:...}} token
    internals: sentences are split on '[.!?] + whitespace', which cannot occur
    inside a token, and protected lines are skipped entirely."""
    needle = (needle or "").strip()[:80]
    if not needle:
        return draft
    lines = draft.splitlines()
    for i, line in enumerate(lines):
        if needle not in line:
            continue
        stripped = line.strip()
        if stripped.startswith(("#", "```", "Table:")):
            continue                      # structural lines are repaired elsewhere
        if stripped.startswith("|"):
            del lines[i]                  # defective table row: drop the row
            return "\n".join(lines)
        sentences = re.split(r"(?<=[.!?])\s+", line)
        kept = [s for s in sentences if needle not in s]
        if len(kept) == len(sentences):
            continue
        lines[i] = " ".join(kept).strip()
        return "\n".join(lines)
    return draft

def apply_repairs(section: str, draft: str, det: Dict[str, Any],
                  llm_ver: Optional[Dict[str, Any]], loop: int) -> Tuple[str, List[Dict[str, Any]]]:
    actions: List[Dict[str, Any]] = []
    new_draft = draft
    gate_by = {g["gate"]: g for g in det["gates"]}

    for f in gate_by.get("fact_token_substitution", {}).get("failures", []):
        new_draft = _remove_sentence_containing(new_draft, "{{FACT:%s}}" % f["key"])
        actions.append({"action": "removed_sentence_with_unknown_token", **f})
    for f in gate_by.get("literal_numbers", {}).get("failures", []):
        new_draft = _remove_sentence_containing(new_draft, f["value"])
        actions.append({"action": "removed_sentence_with_literal_number", **f})
    for f in gate_by.get("period_consistency", {}).get("failures", []):
        new_draft = _remove_sentence_containing(new_draft, f.get("line", "")[:60])
        actions.append({"action": "removed_year_inconsistent_sentence", "key": f.get("key")})
    for f in gate_by.get("cleanliness", {}).get("failures", []):
        new_draft = _remove_sentence_containing(new_draft, f.get("line", "")[:60])
        actions.append({"action": "removed_pipeline_language_sentence", "pattern": f.get("pattern")})
    for f in gate_by.get("no_copy_firewall", {}).get("failures", []):
        new_draft = re.sub(re.escape(f["term"]), "the reference institution", new_draft)
        actions.append({"action": "replaced_forbidden_term", **f})

    for f in gate_by.get("style_mechanics", {}).get("failures", []):
        if f.get("type") == "table_without_caption":
            new_draft = _caption_bare_tables(new_draft)
            actions.append({"action": "inserted_table_caption", "near": f.get("near", "")})
        elif f.get("type") == "heading_not_numbered":
            actions.append({"action": "left_for_rewrite_heading", **f})

    if llm_ver:
        for v in llm_ver["coverage_verdicts"]:
            if v["status"] in ("missing", "unverified"):
                new_draft = patch_missing_requirement(section, new_draft, v["requirement_id"], v.get("gap", ""))
                actions.append({"action": "patched_requirement", "requirement_id": v["requirement_id"],
                                "prior_status": v["status"]})
        for c in llm_ver["claims_check"]["unsupported"]:
            new_draft = _remove_sentence_containing(new_draft, c.get("claim_text", "")[:60])
            actions.append({"action": "removed_unsupported_claim", "claim": c.get("claim_text", "")[:120]})

    write_json(actions, STAGE_DIRS["repairs"] / f"repairs_{SECTION_SLUGS[section]}_loop{loop}.json")
    return new_draft, actions
print("Repair engine ready.")


## 13 · Approval constitution + per-section scoring

In [ ]:
# ============================================================
# CELL 13 — APPROVAL CONSTITUTION + SECTION SCORING
# Approval is OUTPUT-side and fixed in advance:
#   A1 all deterministic gates pass;
#   A2 zero 'missing'/'unverified' verdicts among supported requirements;
#   A3 zero unsupported qualitative claims.
# The style judge is scored and reported, never diluted into other numbers.
# Any relaxation requires an explicit waiver record — there are no silent
# threshold environment variables in this notebook.
# The section score reports each dimension separately; the composite is a
# convenience number computed last and consumed by nothing.
# ============================================================
def approve_section(section: str, det: Dict[str, Any], llm_ver: Dict[str, Any],
                    waivers: Optional[List[Dict[str, Any]]] = None) -> Dict[str, Any]:
    waivers = waivers or []
    failures = []
    if not det["passed"]:
        failures.append({"criterion": "A1_deterministic_gates",
                         "detail": [g["gate"] for g in det["gates"] if not g["passed"]]})
    bad = [v for v in llm_ver["coverage_verdicts"] if v["status"] in ("missing", "unverified")]
    if bad:
        failures.append({"criterion": "A2_output_coverage",
                         "detail": [{"requirement_id": v["requirement_id"], "status": v["status"],
                                     "gap": v["gap"]} for v in bad]})
    if not llm_ver["claims_check"]["passed"]:
        failures.append({"criterion": "A3_qualitative_grounding",
                         "detail": [c.get("claim_text", "")[:120] for c in llm_ver["claims_check"]["unsupported"]]})
    waived = {w.get("criterion") for w in waivers}
    effective = [f for f in failures if f["criterion"] not in waived]
    return {"section_name": section, "approved": not effective, "failures": failures,
            "waivers_applied": waivers, "effective_failures": effective,
            "policy": "A1 gates + A2 output coverage over supported scope + A3 qualitative grounding. "
                      "Unsupported-in-payload requirements are audit-only and cannot fail approval."}

def score_section(section: str, det: Dict[str, Any], llm_ver: Dict[str, Any],
                  approval: Dict[str, Any], loops_used: int) -> Dict[str, Any]:
    counts = Counter(v["status"] for v in llm_ver["coverage_verdicts"])
    n_sup = max(1, len(llm_ver["coverage_verdicts"]))
    output_coverage = round(100 * (counts.get("addressed", 0) + 0.5 * counts.get("partial", 0)) / n_sup, 2)
    lit = next(g for g in det["gates"] if g["gate"] == "literal_numbers")
    tok = next(g for g in det["gates"] if g["gate"] == "fact_token_substitution")
    n_facts = max(1, det["fact_use_count"])
    grounding = round(100 * n_facts / (n_facts + len(lit["failures"]) + len(tok["failures"])), 2)
    style_mech = next(g for g in det["gates"] if g["gate"] == "style_mechanics")
    style_mech_score = 100.0 if style_mech["passed"] else round(
        100 * max(0.0, 1 - len(style_mech["failures"]) / 10), 2)
    applicable_total = len(APPLICABLE_IDS[section])
    payload_readiness = round(100 * len(SUPPORTED[section]) / max(1, applicable_total), 2)
    return {
        "section_name": section,
        "dimensions": {
            "output_coverage_supported_scope_0_100": output_coverage,
            "numeric_grounding_0_100": grounding,
            "style_mechanics_0_100": style_mech_score,
            "style_judge_0_10": llm_ver["style_judge"]["style_score_0_to_10"],
            "qualitative_grounding_passed": llm_ver["claims_check"]["passed"],
        },
        "coverage_counts": dict(counts),
        "supported_requirements": len(SUPPORTED[section]),
        "unsupported_audit_only": len(UNSUPPORTED_REGISTER[section]),
        "payload_readiness_0_100_audit_only": payload_readiness,
        "repair_loops_used": loops_used,
        "approved": approval["approved"],
        "policy": "Dimensions are reported separately. payload_readiness is audit context only and is not "
                  "blended into any quality dimension.",
    }
print("Approval constitution + scoring ready.")


## 14 · LangGraph orchestration (typed state, checkpoints, fallback loop)

In [ ]:
# ============================================================
# CELL 14 — LANGGRAPH ORCHESTRATION (typed state, checkpoints)
# All state lives in SectionState — no stage reads module globals for
# per-section working data. Every node checkpoints its output to disk so a
# run can be resumed and inspected. Falls back to a plain loop if langgraph
# is unavailable; the node functions are IDENTICAL in both paths.
# ============================================================
try:
    from langgraph.graph import StateGraph, END
    from typing_extensions import TypedDict
    _LG = True
except Exception as _exc:
    _LG = False
    from typing import TypedDict  # type: ignore
    print("langgraph unavailable -> plain-loop fallback:", repr(_exc))

class SectionState(TypedDict, total=False):
    section_name: str
    draft_markdown: str
    det: Dict[str, Any]
    llm_ver: Optional[Dict[str, Any]]
    approval: Dict[str, Any]
    score: Dict[str, Any]
    loop: int
    decisions: List[Dict[str, Any]]
    status: str            # "", "approved", "escalated"

def _ckpt(state: SectionState, stage: str) -> None:
    slug = SECTION_SLUGS[state["section_name"]]
    write_json({k: v for k, v in state.items() if k != "det" or stage == "verify"},
               STAGE_DIRS["logs"] / f"ckpt_{slug}_loop{state.get('loop', 0)}_{stage}.json")

def node_write(state: SectionState) -> SectionState:
    section = state["section_name"]
    print(f"[{section}] write (loop {state.get('loop', 0)})")
    obj = write_section_draft(section)
    state["draft_markdown"] = obj.get("draft_markdown", "")
    state.setdefault("decisions", []).append({"loop": state.get("loop", 0), "node": "write"})
    return state

def node_verify(state: SectionState) -> SectionState:
    section = state["section_name"]
    loop = state.get("loop", 0)
    det = run_deterministic_gates(section, state["draft_markdown"])
    state["det"] = det
    write_text(det["final_text"], STAGE_DIRS["drafts"] / f"{SECTION_SLUGS[section]}_loop{loop}.md")
    llm_ver = None
    if det["passed"]:
        print(f"[{section}] gates clean -> LLM verification")
        llm_ver = run_llm_verification(section, det["final_text"])
    else:
        fails = {g['gate']: len(g['failures']) for g in det['gates'] if not g['passed']}
        print(f"[{section}] deterministic failures: {fails}")
    state["llm_ver"] = llm_ver
    ver_out = {"deterministic": {k: det[k] for k in ("passed", "gates", "fact_use_count")},
               "llm_verification": llm_ver}
    write_json(ver_out, STAGE_DIRS["verification"] / f"verification_{SECTION_SLUGS[section]}_loop{loop}.json")
    if llm_ver:
        state["approval"] = approve_section(section, det, llm_ver)
        state["score"] = score_section(section, det, llm_ver, state["approval"], loop)
    else:
        state["approval"] = {"section_name": section, "approved": False,
                             "failures": [{"criterion": "A1_deterministic_gates"}], "effective_failures": [1]}
        state["score"] = {}
    _ckpt(state, "verify")
    return state

def route_after_verify(state: SectionState) -> str:
    if state["approval"]["approved"]:
        return "finalize"
    if state.get("loop", 0) >= MAX_REPAIR_LOOPS:
        return "escalate"
    return "repair"

def node_repair(state: SectionState) -> SectionState:
    section = state["section_name"]
    loop = state.get("loop", 0)
    print(f"[{section}] repair (loop {loop})")
    new_draft, actions = apply_repairs(section, state["draft_markdown"], state["det"], state["llm_ver"], loop)
    state["draft_markdown"] = new_draft
    state["loop"] = loop + 1
    state.setdefault("decisions", []).append({"loop": loop, "node": "repair", "actions": len(actions)})
    return state

def node_finalize(state: SectionState) -> SectionState:
    section = state["section_name"]
    slug = SECTION_SLUGS[section]
    state["status"] = "approved"
    write_text(state["det"]["final_text"], STAGE_DIRS["approved"] / f"approved_{slug}.md")
    write_json({"section_name": section, "status": "approved", "approval": state["approval"],
                "score": state["score"], "decisions": state["decisions"],
                "coverage_verdicts": state["llm_ver"]["coverage_verdicts"],
                "unsupported_register_audit_only": UNSUPPORTED_REGISTER[section]},
               STAGE_DIRS["approved"] / f"approved_{slug}.json")
    print(f"[{section}] APPROVED | coverage {state['score']['dimensions']['output_coverage_supported_scope_0_100']}")
    return state

def node_escalate(state: SectionState) -> SectionState:
    section = state["section_name"]
    slug = SECTION_SLUGS[section]
    state["status"] = "escalated"
    final_text = state.get("det", {}).get("final_text", state.get("draft_markdown", ""))
    write_text(final_text, STAGE_DIRS["approved"] / f"human_review_{slug}.md")
    write_json({"section_name": section, "status": "human_review",
                "approval": state.get("approval", {}), "score": state.get("score", {}),
                "decisions": state.get("decisions", [])},
               STAGE_DIRS["approved"] / f"human_review_{slug}.json")
    print(f"[{section}] ESCALATED to human review after {state.get('loop', 0)} loops "
          f"| unresolved: {[f['criterion'] for f in state.get('approval', {}).get('effective_failures', []) if isinstance(f, dict)]}")
    return state

if _LG:
    _g = StateGraph(SectionState)
    _g.add_node("write", node_write)
    _g.add_node("verify", node_verify)
    _g.add_node("repair", node_repair)
    _g.add_node("finalize", node_finalize)
    _g.add_node("escalate", node_escalate)
    _g.set_entry_point("write")
    _g.add_edge("write", "verify")
    _g.add_conditional_edges("verify", route_after_verify,
                             {"finalize": "finalize", "repair": "repair", "escalate": "escalate"})
    _g.add_edge("repair", "verify")
    _g.add_edge("finalize", END)
    _g.add_edge("escalate", END)
    SECTION_GRAPH = _g.compile()

def run_section(section: str) -> SectionState:
    init: SectionState = {"section_name": section, "loop": 0, "decisions": [], "status": ""}
    if _LG:
        return SECTION_GRAPH.invoke(init, config={"recursion_limit": 8 + 4 * MAX_REPAIR_LOOPS})
    state = node_write(init)
    while True:
        state = node_verify(state)
        route = route_after_verify(state)
        if route == "finalize":
            return node_finalize(state)
        if route == "escalate":
            return node_escalate(state)
        state = node_repair(state)
print(f"Orchestration ready | langgraph={_LG}")


## 15 · Run all sections

In [ ]:
# ============================================================
# CELL 15 — RUN ALL SECTIONS
# ============================================================
SECTION_RESULTS: Dict[str, Dict[str, Any]] = {}
for _section in SECTIONS:
    print("=" * 90)
    SECTION_RESULTS[_section] = run_section(_section)
print("=" * 90)
for _s, _r in SECTION_RESULTS.items():
    print(f"{_s}: {_r.get('status')} | loops {_r.get('loop', 0)}")


## 16 · Assembly → editorial normalisation → FINAL gates → handoff

In [ ]:
# ============================================================
# CELL 16 — ASSEMBLY, EDITORIAL NORMALISATION, FINAL GATES, HANDOFF
# Order is the audit's F7 fix: polish FIRST, then final gates run on the
# exact assembled bytes, then approval is stamped. Nothing mutates the
# report after the final gates. Approved and escalated sections are both
# assembled; escalated ones are marked for review in the manifest only.
# ============================================================
def _renumber_tables_figures(md: str) -> str:
    t = f = 0
    out = []
    for line in md.splitlines():
        if line.strip().startswith("Table:"):
            t += 1
            line = re.sub(r"^(\s*)Table:", rf"\1Table {t}:", line)
        if line.strip() == "```figure":
            f += 1
            line = line + f"\nFigure {f}:"
        out.append(line)
    return "\n".join(out)

def _editorial_normalize(md: str) -> str:
    md = re.sub(r"\n{3,}", "\n\n", md)
    md = re.sub(r"[ \t]+\n", "\n", md)
    # collapse accidental duplicated headings
    seen = set()
    out = []
    for line in md.splitlines():
        if line.startswith("#"):
            key = line.strip()
            if key in seen:
                continue
            seen.add(key)
        out.append(line)
    return "\n".join(out).strip() + "\n"

def assemble_report() -> Dict[str, Any]:
    parts = [f"# {ENTITY_NAME} — IFRS S1 and IFRS S2 Sustainability Disclosures {REPORTING_YEAR}", ""]
    toc = ["## Contents", ""]
    statuses = {}
    for section in SECTIONS:
        slug = SECTION_SLUGS[section]
        approved_path = STAGE_DIRS["approved"] / f"approved_{slug}.md"
        review_path = STAGE_DIRS["approved"] / f"human_review_{slug}.md"
        n = SECTION_NUMBERS[section]
        if approved_path.exists():
            body, statuses[section] = read_text(approved_path), "approved"
        elif review_path.exists():
            body, statuses[section] = read_text(review_path), "human_review"
        else:
            body, statuses[section] = "", "missing"
        toc.append(f"{n}. {section}")
        parts.append(f"# {n} {section}")
        parts.append("")
        parts.append(body)
        parts.append("")
    full = "\n".join(parts[:2] + toc + [""] + parts[2:])
    full = _editorial_normalize(_renumber_tables_figures(full))

    # FINAL gates on the assembled bytes (firewall + cleanliness + literal-number
    # sanity: every numeric token must have been introduced by a fact substitution,
    # which by construction is true iff per-section gates passed and assembly added
    # only heading/table/figure numbers — verified here rather than assumed).
    final_checks = {
        "no_copy_firewall": gate_firewall(full),
        "cleanliness": gate_cleanliness(full),
    }
    final_ok = all(g["passed"] for g in final_checks.values())
    manifest = {
        "entity": ENTITY_NAME, "reporting_year": REPORTING_YEAR,
        "section_status": statuses, "final_gates_passed": final_ok,
        "final_gate_results": final_checks,
        "assembled_at": RUN_STAMP,
        "policy": "No mutation after final gates. Sections marked human_review require sign-off before publication.",
    }
    write_text(full, STAGE_DIRS["assembly"] / "assembled_report.md")
    write_json(manifest, STAGE_DIRS["assembly"] / "assembly_manifest.json")
    print("Assembled report:", STAGE_DIRS["assembly"] / "assembled_report.md",
          "| final gates passed:", final_ok, "| statuses:", statuses)
    return manifest

ASSEMBLY_MANIFEST = assemble_report()


## 17 · Independent evaluation scorecard

In [ ]:
# ============================================================
# CELL 17 — INDEPENDENT EVALUATION SCORECARD
# Reports each dimension separately, from OUTPUT-side signals only.
# payload readiness is shown alongside as audit context, never blended.
# The composite index is computed last and consumed by nothing.
# ============================================================
def _shingles(text: str, n: int = 5) -> set:
    toks = re.findall(r"[a-z0-9]+", text.lower())
    return {" ".join(toks[i:i + n]) for i in range(max(0, len(toks) - n + 1))}

def run_evaluation() -> Dict[str, Any]:
    per_section = {}
    section_texts = {}
    for section in SECTIONS:
        slug = SECTION_SLUGS[section]
        rec = read_json(STAGE_DIRS["approved"] / f"approved_{slug}.json") or \
              read_json(STAGE_DIRS["approved"] / f"human_review_{slug}.json") or {}
        per_section[section] = rec.get("score", {})
        section_texts[section] = read_text(STAGE_DIRS["approved"] / f"approved_{slug}.md",
                                           read_text(STAGE_DIRS["approved"] / f"human_review_{slug}.md"))
    def _mean(vals):
        vals = [v for v in vals if isinstance(v, (int, float))]
        return round(sum(vals) / len(vals), 2) if vals else None
    dims = {
        "output_coverage_supported_scope": _mean([s.get("dimensions", {}).get("output_coverage_supported_scope_0_100") for s in per_section.values()]),
        "numeric_grounding": _mean([s.get("dimensions", {}).get("numeric_grounding_0_100") for s in per_section.values()]),
        "style_mechanics": _mean([s.get("dimensions", {}).get("style_mechanics_0_100") for s in per_section.values()]),
        "style_judge_0_10": _mean([s.get("dimensions", {}).get("style_judge_0_10") for s in per_section.values()]),
    }
    # non-redundancy across sections
    overlaps = []
    names = list(section_texts)
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = _shingles(section_texts[names[i]]), _shingles(section_texts[names[j]])
            if a and b:
                overlaps.append(len(a & b) / min(len(a), len(b)))
    dims["non_redundancy"] = round(100 * (1 - (sum(overlaps) / len(overlaps) if overlaps else 0)), 2)
    dims["payload_coherence_issues"] = len(ARITH_AUDIT["hard_issues"])
    audit_context = {
        "payload_readiness_by_section_audit_only": {
            s: per_section[s].get("payload_readiness_0_100_audit_only") for s in SECTIONS},
        "unsupported_requirements_audit_only": {s: len(UNSUPPORTED_REGISTER[s]) for s in SECTIONS},
        "not_applicable_requirements": {s: len(requirements_by_section[s]) - len(APPLICABLE_IDS[s]) for s in SECTIONS},
    }
    # efficiency from the call log
    calls = [json.loads(l) for l in read_text(_LLM_CALL_LOG).splitlines() if l.strip()]
    efficiency = {"llm_calls": len(calls),
                  "llm_errors": sum(1 for c in calls if "error" in c),
                  "total_latency_s": round(sum(c.get("latency_s", 0) for c in calls), 1)}
    # convenience composite — computed LAST, consumed by NOTHING
    comp_inputs = [dims["output_coverage_supported_scope"], dims["numeric_grounding"],
                   dims["style_mechanics"],
                   (dims["style_judge_0_10"] or 0) * 10, dims["non_redundancy"]]
    composite = round(sum(v for v in comp_inputs if v is not None) / len([v for v in comp_inputs if v is not None]), 2) \
        if any(v is not None for v in comp_inputs) else None
    scorecard = {"dimensions": dims, "per_section": per_section, "audit_context": audit_context,
                 "efficiency": efficiency, "convenience_composite_0_100": composite,
                 "policy": "Dimensions are the result. The composite is descriptive only and is computed "
                           "after all decisions; nothing reads it."}
    write_json(scorecard, STAGE_DIRS["scores"] / "evaluation_scorecard.json")
    print(json.dumps({"dimensions": dims, "composite": composite, "efficiency": efficiency}, indent=1))
    return scorecard

EVALUATION = run_evaluation()


## 18 · Style ablation harness (optional)

In [ ]:
# ============================================================
# CELL 18 — STYLE ABLATION HARNESS (optional; off by default)
# Runs the pipeline under STYLE_ARM in {"none","raw","refined"} into separate
# output directories for comparison. Requires Azure; each arm re-runs
# generation for all sections. Enable with IFRS_RUN_STYLE_ABLATION=1.
# Comparison rule (per sign-off): factual accuracy, grounding, coverage and
# consistency gate the comparison — a style arm that improves style scores
# while worsening any of those is rejected.
# ============================================================
if os.getenv("IFRS_RUN_STYLE_ABLATION", "0") == "1":
    import subprocess, sys
    for arm in ("none", "raw", "refined"):
        env = dict(os.environ)
        env["IFRS_STYLE_ARM"] = arm
        env["IFRS_OUTPUT_DIR"] = str(OUTPUT_DIR.parent / f"ablation_{arm}")
        print(f"--- style arm: {arm} -> {env['IFRS_OUTPUT_DIR']}")
        # Re-executes this notebook headlessly per arm.
        subprocess.run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook",
                        "--execute", os.getenv("IFRS_NOTEBOOK_PATH", "ifrs_report_engine_v2.ipynb"),
                        "--output", f"ablation_{arm}.ipynb"], env=env, check=False)
    print("Ablation runs complete. Compare evaluation_scorecard.json across ablation_* directories.")
else:
    print("Style ablation harness idle (set IFRS_RUN_STYLE_ABLATION=1 to run all three arms).")
